# 03. Temporal Target Selection and History-Regime Construction — Herbal Supplements

This notebook constructs the full downstream-eligible case reserve before final query balancing. The category-specific frozen cutoff and exclusive evaluation-window start is 31 March 2022 at 23:56:10.358 UTC, and the window ends on 31 December 2022 at 23:56:10.358 UTC.

For each user, the procedure examines at most the five most recent reviews in the evaluation window. It reconstructs history separately for each candidate target, rejects a target parent item previously reviewed by the user, discards events occurring after the candidate target, and retains the most recent candidate satisfying the temporal, target-novelty, query-eligibility, and history requirements. At most one target is retained per user.

Two history views are exported. Strict pre-target history contains eligible same-user reviewed-item events before the target timestamp. Training-safe history further restricts these events to the period before the frozen cutoff. Every retained non-cold case must have at least one training-safe prior item in the active item universe.

History regimes are assigned from the effective strict pre-target interaction count: cold = 0, weak = 1–4, and strong ≥ 5. The stored execution exports an eligible reserve of 28,963 cases: 24,590 cold, 3,703 weak, and 670 strong. A deterministic within-regime order supports downstream selection and same-regime replacement.

This notebook does not create the final balanced 1,968-case benchmark. The final quota of 656 cases per regime is fixed downstream only after query evidence has been audited.


In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
# ==== Define Temporal and Sampling Contracts ====
import hashlib
import json
import os
import random
import re
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 300)
pd.set_option("display.width", 200)

CATEGORY_ID = "herbal"
CATEGORY_FOLDER = "herbal_supplements"
CATEGORY_LABEL = "Herbal Supplements"
STAGE = "stage0_user_regime_sampling"
SAMPLING_ARTIFACT_SCHEMA_VERSION = "user_regime_sampling"

PROJECT_ROOT = Path("/content/drive/MyDrive/thesis_recsys/categories") / CATEGORY_FOLDER
os.chdir(PROJECT_ROOT)
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_ITEMS_DIR = DATA_DIR / "processed" / "items"
INTERIM_SAMPLING_DIR = DATA_DIR / "interim" / "user_regime_sampling"
PROCESSED_SAMPLING_DIR = DATA_DIR / "processed" / "user_sampling"
REVIEWS_PATH = RAW_DIR / "reviews_Herbal_Supplements_W2_2019_2022.parquet"
ITEM_SCHEMA_PATH = PROCESSED_ITEMS_DIR / "herbal_item_schema.parquet"

OUTPUT_DIR = INTERIM_SAMPLING_DIR
SAMPLED_QUERY_CASES_PARQUET = OUTPUT_DIR / "herbal_sampled_query_cases.parquet"
SAMPLED_QUERY_CASES_CSV = OUTPUT_DIR / "herbal_sampled_query_cases.csv"
PRIOR_HISTORY_PARQUET = OUTPUT_DIR / "herbal_prior_history.parquet"

SAMPLED_QUERY_CASES_COMPAT_PARQUET = PROCESSED_SAMPLING_DIR / "herbal_user_regime_sample.parquet"
SAMPLED_QUERY_CASES_COMPAT_CSV = PROCESSED_SAMPLING_DIR / "herbal_user_regime_sample.csv"
PRIOR_HISTORY_COMPAT_PARQUET = PROCESSED_SAMPLING_DIR / "herbal_user_prior_review_history.parquet"
TRAIN_PRIOR_HISTORY_PARQUET = PROCESSED_SAMPLING_DIR / "herbal_user_prior_review_history_training.parquet"
TARGET_CASE_METADATA_PARQUET = PROCESSED_SAMPLING_DIR / "herbal_target_case_metadata.parquet"

EVALUATION_WINDOW_MONTHS = 9
TARGET_SELECTION_MODE = "recent_eligible_review_rank_le5"
TARGET_PRIOR_SAME_ITEM_POLICY = "exclude_target_candidates_with_prior_same_parent_asin"
MAX_TARGET_RANK_ALLOWED = 5
TARGET_RANK_CONVENTION = "target_rank_desc is one-based within the evaluation window: 1=latest review, 2=second latest, ..., 5=fifth latest."
MAX_TARGETS_PER_USER = 1

REGIME_ORDER = ["cold", "weak", "strong"]

REQUESTED_SAMPLE_N_BY_REGIME = {
    regime: "deferred_to_query_audit"
    for regime in REGIME_ORDER
}
SAMPLE_N_BY_REGIME = {}
EXPECTED_FINAL_TOTAL_N = None

# Notebook 03 exports the full temporally and history-eligible reserve.
# Query evidence is audited downstream before the final balanced quota is fixed.
EXPECTED_ELIGIBLE_POOL_REGIME_COUNTS = None
EXPECTED_ELIGIBLE_POOL_TOTAL_N = None
DOWNSTREAM_QUERY_BALANCE_N_PER_REGIME = None
DOWNSTREAM_BALANCED_QUERY_REGIME_COUNTS = None
DOWNSTREAM_BALANCED_QUERY_TOTAL_N = None
DOWNSTREAM_QUERY_BALANCING_BASIS = "minimum_query_audit_eligible_regime_supply"

REGIME_DEFINITION = {
    "cold": "prior_history_n == 0",
    "weak": "1 <= prior_history_n <= 4",
    "strong": "prior_history_n >= 5",
}
CATEGORY_SPECIFIC_REGIME_DESIGN = True
MODERATE_REGIME_USED = False

SAMPLING_DECISION_SOURCE = "embedded_final_sampling_decision"
FINAL_WINDOW_CHOICE = f"{EVALUATION_WINDOW_MONTHS}_month"
SAMPLING_DECISION_NOTE = (
    f"Query C-lite relaxed benchmark: final {EVALUATION_WINDOW_MONTHS}-month window, recent_eligible_review_rank_le5 target selection, "
    "rank <= 5 fallback, and full eligible-reserve export after candidate-specific temporal and "
    "history validation. Notebook 03 does not balance regimes or discard reserve cases. "
    "The final quota and deterministic same-regime replacements are resolved only after the "
    "downstream query audit. The sampling eligibility is broad enough to keep query-litable "
    "ingredient/herb-only reviews; ingredient-specific terms should be controlled in downstream query generation."
)

MIN_TARGET_REVIEW_TOKENS = 5
MIN_QUERY_SAFE_TOKEN_COUNT = 2
MIN_QUERY_SAFE_SIGNAL_FAMILIES = 1
MIN_QUERY_SAFE_SIGNAL_TOTAL = 1
STRICT_MIN_QUERY_SAFE_SIGNAL_FAMILIES = 2
STRICT_MIN_QUERY_SAFE_SIGNAL_TOTAL = 3
FINAL_MIN_QUERY_TOKENS = 5
FINAL_MAX_QUERY_TOKENS = 18
FINAL_MIN_QUERY_SAFE_SIGNAL_FAMILIES = 2
FINAL_MIN_QUERY_SAFE_SIGNAL_TOTAL = 3

C_LITE_ELIGIBILITY_FAMILIES = ["benefit_need", "form", "claim_diet"]
MIN_SIGNAL_FAMILIES = 1
MIN_SIGNAL_TOTAL = 1
MIN_SAFE_TOKEN_FALLBACK = 3

DIRECT_CUE_COLS = [
    "brand_primary",
    "brand",
    "brand_norm",
    "manufacturer",
    "family_brand",
    "item_model_number",
    "parent_asin",
    "unit_count",
    "number_of_items",
    "package_dimensions",
]
TITLE_PHRASE_CUE_COLS = ["item_title"]
brand_col = "brand_primary"

REQUIRED_INPUT_PATHS = {
    "reviews": REVIEWS_PATH,
    "item_schema": ITEM_SCHEMA_PATH,
}
missing_inputs = {name: path for name, path in REQUIRED_INPUT_PATHS.items() if not path.exists()}
if missing_inputs:
    raise FileNotFoundError(
        "Missing required input files:\n" + "\n".join(f"- {name}: {path}" for name, path in missing_inputs.items())
    )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_SAMPLING_DIR.mkdir(parents=True, exist_ok=True)
(PROJECT_ROOT / "outputs" / STAGE).mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Output directory:", OUTPUT_DIR)


Project root: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements
Output directory: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/data/interim/user_regime_sampling


In [3]:
# ==== Declare the Shared Sampling Schema ====
COMMON_USER_SAMPLING_FRAMEWORK_VERSION = "common_user_sampling_framework_v1"

COMMON_SAMPLING_COLUMNS = {
    "case_id": "case_id",
    "user_id": "user_id",
    "target_item_id": "target_parent_asin" if CATEGORY_ID == "herbal" else "parent_asin",
    "target_timestamp_ms": "target_timestamp_ms",
    "target_review_datetime": "target_review_datetime" if CATEGORY_ID == "herbal" else "review_datetime",
    "regime": "regime",
    "prior_count": "prior_history_n" if CATEGORY_ID == "herbal" else "prior_review_n",
    "target_rank": "target_rank_desc",
    "target_selection_mode": "target_selection_mode",
    "n_pre_target_interactions": "n_pre_target_interactions",
    "n_pre_target_unique_items": "n_pre_target_unique_items",
    "n_train_safe_interactions": "n_train_safe_interactions",
    "n_train_safe_unique_items": "n_train_safe_unique_items",
    "stage1_profile_available": "stage1_profile_available",
    "stage2_profile_available": "stage2_profile_available",
}

COMMON_PRIOR_HISTORY_COLUMNS = {
    "case_id": "case_id",
    "user_id": "user_id",
    "target_item_id": "target_parent_asin" if CATEGORY_ID == "herbal" else "parent_asin",
    "target_timestamp_ms": "target_timestamp_ms",
    "prior_item_id": "prior_parent_asin" if CATEGORY_ID == "herbal" else "prior_item_id",
    "prior_timestamp_ms": "prior_timestamp_ms",
}

COMMON_USER_SAMPLING_OUTPUT_CONTRACT = {
    "category_id": CATEGORY_ID,
    "category_folder": CATEGORY_FOLDER,
    "artifact_schema_version": SAMPLING_ARTIFACT_SCHEMA_VERSION,
    "framework_version": COMMON_USER_SAMPLING_FRAMEWORK_VERSION,
    "regime_order": REGIME_ORDER,
    "regime_definition": REGIME_DEFINITION,
    "category_specific_regime_design": bool(globals().get("CATEGORY_SPECIFIC_REGIME_DESIGN", CATEGORY_ID == "herbal")),
    "moderate_regime_used": bool("moderate" in REGIME_ORDER),
    "sampling_columns": COMMON_SAMPLING_COLUMNS,
    "prior_history_columns": COMMON_PRIOR_HISTORY_COLUMNS,
    "policy_note": "Sample-size quotas, target windows, rank limits, and eligibility thresholds remain category-specific; only the structural output contract is shared.",
    "notebook_03_output_role": "full_eligible_reserve_before_query_audit_and_balance",
    "expected_eligible_pool_regime_counts": EXPECTED_ELIGIBLE_POOL_REGIME_COUNTS,
    "expected_eligible_pool_total_n": EXPECTED_ELIGIBLE_POOL_TOTAL_N,
    "downstream_query_balance_n_per_regime": DOWNSTREAM_QUERY_BALANCE_N_PER_REGIME,
    "downstream_balanced_query_regime_counts": DOWNSTREAM_BALANCED_QUERY_REGIME_COUNTS,
    "downstream_balanced_query_total_n": DOWNSTREAM_BALANCED_QUERY_TOTAL_N,
    "downstream_query_balancing_basis": DOWNSTREAM_QUERY_BALANCING_BASIS,
}

# Shared critical sampling contract. This block is identical across categories.
COMMON_CRITICAL_SAMPLING_CONTRACT = {
    "version": "fresh_unified_v2",
    "candidate_history": "recomputed_for_each_target_candidate",
    "strict_history_predicate": "prior_timestamp_ms < target_timestamp_ms",
    "training_history_predicate": "prior_timestamp_ms < min(target_timestamp_ms, training_cutoff_ms)",
    "target_item_policy": "exclude_target_parent_from_both_histories_and_reject_previously_reviewed_target",
    "regime_source": "n_pre_target_interactions",
    "target_selection": "most_recent_eligible_target_within_five_most_recent_window_reviews",
    "one_target_per_user": True,
    "non_cold_stage1_requirement": "at_least_one_training_safe_prior_item",
}
COMMON_CRITICAL_SAMPLING_CONTRACT_SHA256 = hashlib.sha256(
    json.dumps(
        COMMON_CRITICAL_SAMPLING_CONTRACT,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")
).hexdigest()
EXPECTED_COMMON_CRITICAL_SAMPLING_CONTRACT_SHA256 = "13d580fcdc9e2d5bf25c48090d05681ed6b7b2b894a42421541c75763a342259"
if COMMON_CRITICAL_SAMPLING_CONTRACT_SHA256 != EXPECTED_COMMON_CRITICAL_SAMPLING_CONTRACT_SHA256:
    raise RuntimeError("Common critical sampling contract hash mismatch.")


In [4]:
# ==== Validate Fixed Sampling Parameters ====
if EVALUATION_WINDOW_MONTHS != 9:
    raise ValueError("This Herbal C-lite sampling notebook expects EVALUATION_WINDOW_MONTHS = 9.")
if TARGET_SELECTION_MODE != "recent_eligible_review_rank_le5":
    raise ValueError("This Herbal C-lite sampling notebook expects TARGET_SELECTION_MODE = recent_eligible_review_rank_le5.")
if MAX_TARGET_RANK_ALLOWED != 5:
    raise ValueError("This Herbal C-lite sampling notebook expects MAX_TARGET_RANK_ALLOWED = 5.")
if set(REQUESTED_SAMPLE_N_BY_REGIME) != set(REGIME_ORDER):
    raise ValueError("REQUESTED_SAMPLE_N_BY_REGIME must define every regime.")
if MIN_TARGET_REVIEW_TOKENS != 5 or MIN_QUERY_SAFE_TOKEN_COUNT != 2:
    raise ValueError("C-lite sampling expects MIN_TARGET_REVIEW_TOKENS=5 and MIN_QUERY_SAFE_TOKEN_COUNT=2.")
if FINAL_MIN_QUERY_TOKENS != 5 or FINAL_MAX_QUERY_TOKENS != 18:
    raise ValueError("Final query constraints expect FINAL_MIN_QUERY_TOKENS=5 and FINAL_MAX_QUERY_TOKENS=18.")
if FINAL_MIN_QUERY_SAFE_SIGNAL_FAMILIES != 2 or FINAL_MIN_QUERY_SAFE_SIGNAL_TOTAL != 3:
    raise ValueError("Final query constraints expect FINAL_MIN_QUERY_SAFE_SIGNAL_FAMILIES=2 and FINAL_MIN_QUERY_SAFE_SIGNAL_TOTAL=3.")
if C_LITE_ELIGIBILITY_FAMILIES != ["benefit_need", "form", "claim_diet"]:
    raise ValueError("C-lite diagnostic families must prioritize benefit/form/claim signals.")
if MIN_SAFE_TOKEN_FALLBACK != 3:
    raise ValueError("C-lite sampling expects MIN_SAFE_TOKEN_FALLBACK = 3.")
if MAX_TARGET_RANK_ALLOWED is not None and MAX_TARGET_RANK_ALLOWED < 2:
    raise ValueError("MAX_TARGET_RANK_ALLOWED must be at least 2 for recent eligible fallback sampling.")

print("Evaluation window months:", EVALUATION_WINDOW_MONTHS)
print("Target selection mode:", TARGET_SELECTION_MODE)
print("Max target rank allowed:", MAX_TARGET_RANK_ALLOWED)
print("Requested sample sizes by regime:", REQUESTED_SAMPLE_N_BY_REGIME)
print("C-lite safe token fallback:", MIN_SAFE_TOKEN_FALLBACK)


Evaluation window months: 9
Target selection mode: recent_eligible_review_rank_le5
Max target rank allowed: 5
Requested sample sizes by regime: {'cold': 'deferred_to_query_audit', 'weak': 'deferred_to_query_audit', 'strong': 'deferred_to_query_audit'}
C-lite safe token fallback: 3


In [5]:
# ==== Define Sampling Helpers ====
def normalize_space(value):
    if value is None or pd.isna(value):
        return ''
    return re.sub(r'\s+', ' ', str(value).replace('\n', ' ').replace('\t', ' ')).strip()


def to_review_timestamp_ms(series: pd.Series) -> pd.Series:
    if pd.api.types.is_datetime64_any_dtype(series):
        parsed = pd.to_datetime(series, utc=True, errors='coerce')
        return (parsed.astype('int64') // 1_000_000).where(parsed.notna(), np.nan)
    numeric = pd.to_numeric(series, errors='coerce')
    if numeric.dropna().empty:
        parsed = pd.to_datetime(series, utc=True, errors='coerce')
        return (parsed.astype('int64') // 1_000_000).where(parsed.notna(), np.nan)
    if numeric.dropna().max() < 10**11:
        numeric = numeric * 1000
    return numeric


def tokenize_text(text):
    return re.findall(r"[A-Za-z0-9']+", normalize_space(text).lower())


def normalize_bool_flag(value):
    if value is None or pd.isna(value):
        return False
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (int, np.integer, float, np.floating)):
        return bool(value)
    return normalize_space(value).lower() in {'true', 'yes', 'y', '1', 'discontinued'}


def regime_from_prior_count(prior_n):
    n = int(0 if pd.isna(prior_n) else prior_n)
    if n == 0:
        return 'cold'
    if 1 <= n <= 4:
        return 'weak'
    if n >= 5:
        return 'strong'
    return 'other'


In [6]:
# ==== Load Reviews and Required Item Fields ====
review_columns = ['user_id', 'parent_asin', 'timestamp', 'title', 'text', 'verified_purchase']
item_columns = [
    'parent_asin', 'title', 'brand_primary', 'brand_norm', 'brand', 'manufacturer',
    'sub_category', 'sub_category_norm', 'item_form', 'item_form_norm', 'product_benefits',
    'active_ingredients', 'special_ingredients', 'primary_supplement_type', 'diet_type',
    'material_feature', 'flavor', 'scent', 'family_brand', 'family_sub_category',
    'family_item_form', 'family_benefit_function', 'family_usage_need',
    'family_ingredient_composition', 'family_claims_diet', 'family_flavor',
    'family_usage_target', 'unit_count', 'number_of_items', 'package_dimensions',
    'item_model_number', 'is_discontinued', 'is_discontinued_norm',
]

reviews_raw = pd.read_parquet(REVIEWS_PATH, columns=review_columns)
item_schema = pd.read_parquet(ITEM_SCHEMA_PATH, columns=item_columns)

reviews = pd.DataFrame({
    'user_id': reviews_raw['user_id'].astype(str).map(normalize_space),
    'parent_asin': reviews_raw['parent_asin'].astype(str).map(normalize_space),
    'review_timestamp_ms': to_review_timestamp_ms(reviews_raw['timestamp']),
    'review_title_text': reviews_raw['title'].map(normalize_space),
    'review_body_text': reviews_raw['text'].map(normalize_space),
    'verified_purchase': reviews_raw['verified_purchase'],
})
reviews['review_datetime'] = pd.to_datetime(reviews['review_timestamp_ms'], unit='ms', utc=True, errors='coerce')
reviews['review_text'] = (reviews['review_title_text'] + ' ' + reviews['review_body_text']).map(normalize_space)
reviews = reviews[reviews['review_datetime'].notna()].copy()
reviews = reviews[reviews['user_id'].ne('') & reviews['parent_asin'].ne('')].copy()
reviews['review_timestamp_ms'] = pd.to_numeric(reviews['review_timestamp_ms'], errors='coerce')
reviews = reviews.dropna(subset=['review_timestamp_ms']).copy()

item_schema = item_schema[item_schema['parent_asin'].notna()].copy()
item_schema['parent_asin'] = item_schema['parent_asin'].astype(str).map(normalize_space)
item_schema = item_schema[item_schema['parent_asin'].ne('')].copy()
item_schema = item_schema.sort_values('parent_asin', kind='mergesort').drop_duplicates('parent_asin', keep='first')
item_schema = item_schema.rename(columns={'title': 'item_title'})
item_schema['item_is_discontinued'] = item_schema['is_discontinued_norm'].map(normalize_bool_flag) | item_schema['is_discontinued'].map(normalize_bool_flag)

MAX_REVIEW_DATE = reviews['review_datetime'].max()
if pd.isna(MAX_REVIEW_DATE):
    raise RuntimeError('Could not compute MAX_REVIEW_DATE from reviews.')
print('Review rows:', len(reviews))
print('Item rows:', len(item_schema))
print('MAX_REVIEW_DATE:', MAX_REVIEW_DATE)


Review rows: 593363
Item rows: 27407
MAX_REVIEW_DATE: 2022-12-31 23:56:10.358000+00:00


In [7]:
# ==== Enumerate Up to Five Recent Target Candidates per User ====
EVALUATION_WINDOW_END = MAX_REVIEW_DATE
EVALUATION_WINDOW_START = EVALUATION_WINDOW_END - pd.DateOffset(months=EVALUATION_WINDOW_MONTHS)
# Stage 1 training-safe history is bounded by the frozen cutoff.
# The cutoff is also the exclusive start of the nine-month evaluation window.
TRAIN_REVIEW_CUTOFF_EXCLUSIVE = EVALUATION_WINDOW_START
if TRAIN_REVIEW_CUTOFF_EXCLUSIVE != EVALUATION_WINDOW_START:
    raise RuntimeError(
        "The historical-review cutoff must equal the evaluation-window start."
    )
window_reviews = reviews[
    (reviews['review_datetime'] > EVALUATION_WINDOW_START)
    & (reviews['review_datetime'] <= EVALUATION_WINDOW_END)
].copy()
if window_reviews.empty:
    raise RuntimeError('Evaluation window has no reviews.')
if window_reviews['review_datetime'].le(EVALUATION_WINDOW_START).any():
    raise RuntimeError('Evaluation window must be start-exclusive.')
if window_reviews['review_datetime'].gt(EVALUATION_WINDOW_END).any():
    raise RuntimeError('Evaluation window must be end-inclusive with no rows after the end.')

target_candidates = (
    window_reviews.reset_index(drop=False).rename(columns={'index': 'review_row_id'})
    .sort_values(['user_id', 'review_timestamp_ms', 'parent_asin', 'review_row_id'], ascending=[True, False, True, True], kind='mergesort')
    .copy()
)
target_candidates['target_rank_desc'] = target_candidates.groupby('user_id').cumcount() + 1
if MAX_TARGET_RANK_ALLOWED is not None:
    target_candidates = target_candidates[target_candidates['target_rank_desc'].le(MAX_TARGET_RANK_ALLOWED)].copy()

# Same-item novelty is computed later from strictly earlier timestamps for
# each candidate. Do not use row-order cumcount because equal-time events are
# not prior evidence under the strict temporal contract.
if MAX_TARGET_RANK_ALLOWED is not None and not target_candidates['target_rank_desc'].le(MAX_TARGET_RANK_ALLOWED).all():
    raise RuntimeError(f'target_rank_desc must be <= {MAX_TARGET_RANK_ALLOWED}.')
print('Users in window:', window_reviews['user_id'].nunique())
print(f'Rank <= {MAX_TARGET_RANK_ALLOWED} target candidates:', len(target_candidates))
print('Prior same-item review counts are computed after candidate-specific strict-history reconstruction.')
print('Target prior same-item policy:', TARGET_PRIOR_SAME_ITEM_POLICY)


Users in window: 77615
Rank <= 5 target candidates: 90488
Prior same-item review counts are computed after candidate-specific strict-history reconstruction.
Target prior same-item policy: exclude_target_candidates_with_prior_same_parent_asin


In [8]:
# ==== Join Target-Item Metadata for Eligibility Checks ====
target_candidates = target_candidates.merge(item_schema, on='parent_asin', how='left', indicator='item_merge_status')
target_candidates['item_matched'] = target_candidates['item_merge_status'].eq('both')
target_candidates['item_is_discontinued'] = target_candidates['item_is_discontinued'].fillna(False).astype(bool)
print('Matched target candidates:', int(target_candidates['item_matched'].sum()))


Matched target candidates: 90488


In [9]:
# ==== Evaluate Review-to-Query Eligibility ====
ASIN_PATTERN = re.compile(r'\bb0[a-z0-9]{8}\b', re.IGNORECASE)
PACKAGE_IDENTIFIER_PATTERN = re.compile(
    r'\b('
    r'\d+(?:\.\d+)?\s?(?:mg|mcg|iu|g|gram|grams|ml|oz|fl\.?\s?oz|ct|count|serving|servings|pack|packs|bottle|bottles)'
    r'|\d+\s?(?:x|times|per)\s?(?:day|daily|week|month)?'
    r'|\d+\s?(?:capsule|capsules|tablet|tablets|softgel|softgels|gummy|gummies|drop|drops)'
    r'|(?:capsule|capsules|tablet|tablets|softgel|softgels|gummy|gummies|serving|servings)\s?\d+'
    r'|asin|sku|upc|barcode|seller|manufacturer'
    r')\b',
    re.IGNORECASE,
)
DIRECT_CUE_STOPWORDS = {
    'the', 'and', 'or', 'for', 'with', 'from', 'product', 'supplement', 'supplements',
    'herbal', 'health', 'capsule', 'capsules', 'tablet', 'tablets', 'softgel', 'softgels',
    'gummy', 'gummies', 'powder', 'liquid', 'extract', 'tea', 'drops', 'organic', 'natural',
}
SIGNAL_PATTERNS = {
    'ingredient_or_herb': [r'\bturmeric\b', r'\bcurcumin\b', r'\bginger\b', r'\belderberry\b', r'\bashwagandha\b', r'\bmushrooms?\b', r"\blion'?s mane\b", r'\breishi\b', r'\bchaga\b', r'\bmilk thistle\b', r'\bechinacea\b', r'\bginseng\b', r'\bcranberry\b', r'\bpeppermint\b', r'\bchamomile\b', r'\bvalerian\b', r'\bgarlic\b', r'\bberberine\b', r'\bmaca\b', r'\bmoringa\b', r'\bsaw palmetto\b', r'\bblack seed\b'],
    'benefit_need': [r'\bimmune\b', r'\bimmunity\b', r'\bsleep\b', r'\bstress\b', r'\bcalm\b', r'\brelax\w*\b', r'\bdigestion\b', r'\bdigest\w*\b', r'\bstomach\b', r'\bgut\b', r'\benergy\b', r'\bfocus\b', r'\bmemory\b', r'\bjoint\b', r'\binflammation\b', r'\bliver\b', r'\bdetox\b', r'\bthroat\b', r'\burinary\b', r'\bbladder\b'],
    'form': [r'\bcapsules?\b', r'\btablets?\b', r'\bsoftgels?\b', r'\bgumm(?:y|ies)\b', r'\bpowder\b', r'\btea\b', r'\bliquid extract\b', r'\btincture\b', r'\bdrops?\b'],
    'claim_diet': [r'\borganic\b', r'\bvegan\b', r'\bvegetarian\b', r'\bnon[- ]?gmo\b', r'\bgluten[- ]?free\b', r'\bsugar[- ]?free\b', r'\bcaffeine[- ]?free\b', r'\bdairy[- ]?free\b', r'\bsoy[- ]?free\b'],
    'flavor': [r'\bberry\b', r'\blemon\b', r'\borange\b', r'\bmint\b', r'\bhoney\b'],
}


def cue_terms_from_value(value):
    text = normalize_space(value).lower()
    if not text:
        return []
    terms = [text]
    terms.extend([tok for tok in tokenize_text(text) if len(tok) >= 4 and tok not in DIRECT_CUE_STOPWORDS])
    return list(dict.fromkeys([term for term in terms if term]))


def title_phrase_terms_from_value(value):
    text = normalize_space(value).lower()
    if not text:
        return []
    token_count = len(tokenize_text(text))
    if token_count >= 3 and len(text) >= 12:
        return [text]
    return []


def scrub_cue_terms(scrubbed, cue_terms):
    cue_hits = 0
    for term in sorted(set(cue_terms), key=len, reverse=True):
        term = normalize_space(term).lower()
        if len(term) < 4:
            continue
        scrubbed, n = re.subn(r'(?<![a-z0-9])' + re.escape(term) + r'(?![a-z0-9])', ' ', scrubbed)
        cue_hits += int(n)
        compact_term = re.sub(r'[^a-z0-9]+', '', term)
        if len(compact_term) >= 6:
            scrubbed, n = re.subn(r'(?<![a-z0-9])' + re.escape(compact_term) + r'(?![a-z0-9])', ' ', scrubbed)
            cue_hits += int(n)
    return scrubbed, cue_hits


def remove_direct_item_cues(text, row):
    scrubbed = normalize_space(text).lower()
    cue_terms = []
    title_phrase_terms = []

    for col in DIRECT_CUE_COLS:
        if col in row.index:
            cue_terms.extend(cue_terms_from_value(row.get(col, '')))

    for col in TITLE_PHRASE_CUE_COLS:
        if col in row.index:
            title_phrase_terms.extend(title_phrase_terms_from_value(row.get(col, '')))

    scrubbed, direct_cue_hits = scrub_cue_terms(scrubbed, cue_terms)
    scrubbed, title_phrase_hits = scrub_cue_terms(scrubbed, title_phrase_terms)
    scrubbed, asin_hits = ASIN_PATTERN.subn(' ', scrubbed)
    scrubbed, package_hits = PACKAGE_IDENTIFIER_PATTERN.subn(' ', scrubbed)
    return normalize_space(scrubbed), int(direct_cue_hits + title_phrase_hits + asin_hits + package_hits)


def extract_signal_counts(query_safe_text):
    lowered = normalize_space(query_safe_text).lower()
    family_hits = {
        family: int(any(re.search(pattern, lowered) for pattern in patterns))
        for family, patterns in SIGNAL_PATTERNS.items()
    }
    signal_total_by_family = {
        family: sum(int(bool(re.search(pattern, lowered))) for pattern in patterns)
        for family, patterns in SIGNAL_PATTERNS.items()
    }
    signal_total = int(sum(signal_total_by_family.values()))
    c_lite_family_count = int(sum(family_hits.get(family, 0) for family in C_LITE_ELIGIBILITY_FAMILIES))
    c_lite_signal_total = int(sum(signal_total_by_family.get(family, 0) for family in C_LITE_ELIGIBILITY_FAMILIES))
    return int(sum(family_hits.values())), signal_total, family_hits, c_lite_family_count, c_lite_signal_total


def query_convertibility_diagnostics(row):
    review_text = normalize_space(row.get('review_text', ''))
    query_safe_text, direct_item_cue_hit_count = remove_direct_item_cues(review_text, row)
    review_tokens = len(tokenize_text(review_text))
    safe_tokens = len(tokenize_text(query_safe_text))
    family_count, signal_total, family_hits, c_lite_family_count, c_lite_signal_total = extract_signal_counts(query_safe_text)
    c_lite_signal_pass = (
        (c_lite_family_count >= MIN_SIGNAL_FAMILIES and c_lite_signal_total >= MIN_SIGNAL_TOTAL)
        or safe_tokens >= MIN_SAFE_TOKEN_FALLBACK
    )
    primary_pass = (
        review_text != ''
        and review_tokens >= MIN_TARGET_REVIEW_TOKENS
        and bool(row.get('item_matched', False))
        and not bool(row.get('item_is_discontinued', False))
        and safe_tokens >= MIN_QUERY_SAFE_TOKEN_COUNT
        and family_count >= MIN_QUERY_SAFE_SIGNAL_FAMILIES
        and signal_total >= MIN_QUERY_SAFE_SIGNAL_TOTAL
        and c_lite_signal_pass
    )
    strict_pass = primary_pass and family_count >= STRICT_MIN_QUERY_SAFE_SIGNAL_FAMILIES and signal_total >= STRICT_MIN_QUERY_SAFE_SIGNAL_TOTAL
    if review_text == '':
        reason = 'missing_review_text'
    elif review_tokens < MIN_TARGET_REVIEW_TOKENS:
        reason = 'short_review'
    elif not bool(row.get('item_matched', False)):
        reason = 'missing_item_metadata'
    elif bool(row.get('item_is_discontinued', False)):
        reason = 'discontinued_item'
    elif safe_tokens < MIN_QUERY_SAFE_TOKEN_COUNT:
        reason = 'low_query_safe_token_count'
    elif family_count < MIN_QUERY_SAFE_SIGNAL_FAMILIES:
        reason = 'low_signal_family_count'
    elif signal_total < MIN_QUERY_SAFE_SIGNAL_TOTAL:
        reason = 'low_signal_total_count'
    elif not c_lite_signal_pass:
        reason = 'low_signal_and_token_fallback'
    else:
        reason = 'eligible'
    out = {
        'query_safe_text': query_safe_text,
        'query_convertible_flag': int(primary_pass),
        'strict_diagnostic_convertible_flag': int(strict_pass),
        'query_convertibility_failure_reason': reason,
        'query_safe_token_count': int(safe_tokens),
        'query_safe_signal_family_count': int(family_count),
        'query_safe_signal_total_count': int(signal_total),
        'c_lite_query_safe_signal_family_count': int(c_lite_family_count),
        'c_lite_query_safe_signal_total_count': int(c_lite_signal_total),
        'c_lite_token_fallback_pass': int(safe_tokens >= MIN_SAFE_TOKEN_FALLBACK),
        'c_lite_signal_pass': int(c_lite_signal_pass),
        'direct_item_cue_hit_count': int(direct_item_cue_hit_count),
    }
    out.update({f'query_safe_has_{family}': int(value) for family, value in family_hits.items()})
    return pd.Series(out)


diagnostics = target_candidates.apply(query_convertibility_diagnostics, axis=1)
target_candidates = pd.concat([target_candidates, diagnostics], axis=1)
print(f'Query-convertible rank <= {MAX_TARGET_RANK_ALLOWED} candidates:', int(target_candidates['query_convertible_flag'].sum()))


Query-convertible rank <= 5 candidates: 35246


In [10]:
# ==== Rebuild Candidate-Specific Histories and Select One Target ====
def compute_candidate_history_metrics(all_reviews: pd.DataFrame, candidates: pd.DataFrame) -> pd.DataFrame:
    history = (
        all_reviews[["user_id", "parent_asin", "review_timestamp_ms"]]
        .dropna(subset=["review_timestamp_ms"])
        .sort_values(["user_id", "review_timestamp_ms", "parent_asin"], kind="mergesort")
        .copy()
    )
    history_by_user = {
        user_id: group.copy()
        for user_id, group in history.groupby("user_id", sort=False)
    }
    training_cutoff_ms = int(TRAIN_REVIEW_CUTOFF_EXCLUSIVE.value // 1_000_000)
    rows = []

    required_cols = ["review_row_id", "user_id", "parent_asin", "review_timestamp_ms"]
    missing_cols = [col for col in required_cols if col not in candidates.columns]
    if missing_cols:
        raise RuntimeError(f"Target candidates are missing history keys: {missing_cols}")
    if candidates["review_row_id"].duplicated().any():
        raise RuntimeError("review_row_id must be unique before candidate history computation.")

    for row in candidates[required_cols].itertuples(index=False):
        user_history = history_by_user.get(row.user_id)
        target_timestamp_ms = int(row.review_timestamp_ms)
        target_item_id = str(row.parent_asin)

        if user_history is None or user_history.empty:
            strictly_earlier = history.iloc[0:0]
        else:
            strictly_earlier = user_history[
                user_history["review_timestamp_ms"].lt(target_timestamp_ms)
            ]

        prior_same_item = strictly_earlier[
            strictly_earlier["parent_asin"].astype(str).eq(target_item_id)
        ]
        pre_target_history = strictly_earlier[
            ~strictly_earlier["parent_asin"].astype(str).eq(target_item_id)
        ]

        effective_training_cutoff_ms = min(target_timestamp_ms, training_cutoff_ms)
        train_safe_history = pre_target_history[
            pre_target_history["review_timestamp_ms"].lt(effective_training_cutoff_ms)
        ]

        n_pre_target_interactions = int(len(pre_target_history))
        n_pre_target_unique_items = int(pre_target_history["parent_asin"].nunique())
        n_train_safe_interactions = int(len(train_safe_history))
        n_train_safe_unique_items = int(train_safe_history["parent_asin"].nunique())

        rows.append({
            "review_row_id": int(row.review_row_id),
            "n_pre_target_interactions": n_pre_target_interactions,
            "n_pre_target_unique_items": n_pre_target_unique_items,
            "n_train_safe_interactions": n_train_safe_interactions,
            "n_train_safe_unique_items": n_train_safe_unique_items,
            "stage1_profile_available": bool(n_train_safe_unique_items >= 1),
            "stage2_profile_available": bool(n_pre_target_unique_items >= 1),
            "prior_same_item_review_n": int(len(prior_same_item)),
        })

    return pd.DataFrame(rows)


candidate_history_metrics = compute_candidate_history_metrics(reviews, target_candidates)
target_candidates = target_candidates.merge(
    candidate_history_metrics,
    on="review_row_id",
    how="left",
    validate="one_to_one",
)
count_columns = [
    "n_pre_target_interactions",
    "n_pre_target_unique_items",
    "n_train_safe_interactions",
    "n_train_safe_unique_items",
    "prior_same_item_review_n",
]
target_candidates[count_columns] = target_candidates[count_columns].fillna(0).astype(int)
target_candidates["stage1_profile_available"] = (
    target_candidates["stage1_profile_available"].fillna(False).astype(bool)
)
target_candidates["stage2_profile_available"] = (
    target_candidates["stage2_profile_available"].fillna(False).astype(bool)
)
target_candidates["target_item_repeat_prior_flag"] = (
    target_candidates["prior_same_item_review_n"].gt(0).astype(int)
)

# Compatibility aliases retain strict pre-target history, not training-cutoff history.
target_candidates["prior_review_n"] = target_candidates["n_pre_target_interactions"].astype(int)
target_candidates["unique_prior_item_count"] = target_candidates["n_pre_target_unique_items"].astype(int)
target_candidates["prior_history_n"] = target_candidates["n_pre_target_interactions"].astype(int)
target_candidates["prior_item_n"] = target_candidates["n_pre_target_unique_items"].astype(int)
target_candidates["strict_training_prior_rows"] = target_candidates["n_train_safe_interactions"].astype(int)
target_candidates["strict_training_prior_unique_items"] = target_candidates["n_train_safe_unique_items"].astype(int)
target_candidates["raw_pre_target_prior_history_n"] = target_candidates["n_pre_target_interactions"].astype(int)
target_candidates["raw_pre_target_prior_item_n"] = target_candidates["n_pre_target_unique_items"].astype(int)
target_candidates["regime"] = target_candidates["n_pre_target_interactions"].map(regime_from_prior_count)
target_candidates["stage1_sampling_eligible"] = (
    target_candidates["regime"].eq("cold")
    | target_candidates["stage1_profile_available"]
)

if (
    target_candidates["n_train_safe_interactions"]
    > target_candidates["n_pre_target_interactions"]
).any():
    raise RuntimeError("Training-safe history cannot exceed strict pre-target history.")
if (
    target_candidates["n_train_safe_unique_items"]
    > target_candidates["n_pre_target_unique_items"]
).any():
    raise RuntimeError("Training-safe unique items cannot exceed strict pre-target unique items.")

rank_limit_mask = (
    pd.Series(True, index=target_candidates.index)
    if MAX_TARGET_RANK_ALLOWED is None
    else target_candidates["target_rank_desc"].le(MAX_TARGET_RANK_ALLOWED)
)
eligible_pool = target_candidates[
    rank_limit_mask
    & target_candidates["query_convertible_flag"].eq(1)
    & target_candidates["target_item_repeat_prior_flag"].eq(0)
    & target_candidates["stage1_sampling_eligible"]
].copy()
eligible_supply_by_regime_before_user_dedup = (
    eligible_pool.groupby("regime")["user_id"]
    .nunique()
    .reindex(REGIME_ORDER, fill_value=0)
    .astype(int)
    .to_dict()
)
eligible_candidate_rows_by_regime_before_user_dedup = (
    eligible_pool["regime"]
    .value_counts()
    .reindex(REGIME_ORDER, fill_value=0)
    .astype(int)
    .to_dict()
)
users_with_no_eligible_review_in_top5 = sorted(
    set(target_candidates["user_id"].dropna().astype(str))
    - set(eligible_pool["user_id"].dropna().astype(str))
)
selected_targets = (
    eligible_pool
    .sort_values(
        ["user_id", "target_rank_desc", "review_timestamp_ms", "parent_asin", "review_row_id"],
        ascending=[True, True, False, True, True],
        kind="mergesort",
    )
    .groupby("user_id", as_index=False, sort=False)
    .head(1)
    .copy()
)
selected_targets["target_selection_mode"] = TARGET_SELECTION_MODE
selected_targets["target_parent_asin"] = selected_targets["parent_asin"]
selected_targets["target_review_row_id"] = selected_targets["review_row_id"]
selected_targets["target_timestamp_ms"] = selected_targets["review_timestamp_ms"].astype("int64")
selected_targets["target_review_datetime"] = selected_targets["review_datetime"]
selected_targets_by_rank = (
    selected_targets["target_rank_desc"]
    .value_counts()
    .reindex(range(1, MAX_TARGET_RANK_ALLOWED + 1), fill_value=0)
    .astype(int)
    .to_dict()
)
eligible_supply_by_regime_after_user_dedup = (
    selected_targets["regime"]
    .value_counts()
    .reindex(REGIME_ORDER, fill_value=0)
    .astype(int)
    .to_dict()
)

if selected_targets["user_id"].duplicated().any():
    raise RuntimeError("Selected target users must be unique.")
if not selected_targets["regime"].eq(
    selected_targets["n_pre_target_interactions"].map(regime_from_prior_count)
).all():
    raise RuntimeError("Regime must be derived from strict pre-target history for each target candidate.")

bad_non_cold_stage1 = selected_targets[
    selected_targets["regime"].ne("cold")
    & selected_targets["n_train_safe_unique_items"].lt(1)
]
print("Selected non-cold targets with zero training-safe prior items:", len(bad_non_cold_stage1))
if len(bad_non_cold_stage1):
    display(bad_non_cold_stage1[["user_id", "parent_asin", "review_timestamp_ms", "regime", "n_pre_target_interactions", "n_train_safe_interactions"]].head(20))

print("Selected eligible targets:", len(selected_targets))
print("Eligible targets after target-item and Stage 1 history filtering:", len(eligible_pool))
print(
    f"Rank <= {MAX_TARGET_RANK_ALLOWED} target candidates with strict prior same-item review:",
    int(target_candidates["target_item_repeat_prior_flag"].sum()),
)
print("Target rank convention:", TARGET_RANK_CONVENTION)
print("Selected targets by rank:", selected_targets_by_rank)
print("Eligible supply by regime before user deduplication:", eligible_supply_by_regime_before_user_dedup)
print("Eligible supply by regime after user deduplication:", eligible_supply_by_regime_after_user_dedup)
print("Users with no eligible review among top five:", len(users_with_no_eligible_review_in_top5))
if users_with_no_eligible_review_in_top5:
    print("Users with no eligible review among top five, first 20:", users_with_no_eligible_review_in_top5[:20])
print("Non-cold candidates rejected for zero training-safe prior items:", int(
    (
        target_candidates["regime"].ne("cold")
        & ~target_candidates["stage1_profile_available"]
    ).sum()
))

Selected non-cold targets with zero training-safe prior items: 0
Selected eligible targets: 28963
Eligible targets after target-item and Stage 1 history filtering: 31545
Rank <= 5 target candidates with strict prior same-item review: 191
Target rank convention: target_rank_desc is one-based within the evaluation window: 1=latest review, 2=second latest, ..., 5=fifth latest.
Selected targets by rank: {1: 26868, 2: 1462, 3: 392, 4: 135, 5: 106}
Eligible supply by regime before user deduplication: {'cold': 24590, 'weak': 3809, 'strong': 670}
Eligible supply by regime after user deduplication: {'cold': 24590, 'weak': 3703, 'strong': 670}
Users with no eligible review among top five: 48652
Users with no eligible review among top five, first 20: ['AE227AY2ASJOP4KGTHAPXJPJZBTQ', 'AE227WO2REFLIYSYRFA66KSKBOGQ', 'AE22BBS3LOFWEU3MM6ED4TJNP63Q', 'AE22BKDMQZDCLZYADAQB7TLR4EGQ', 'AE22JTEFLUGSSSQM4CIPEOBSCDZA', 'AE22KNU5IL43GJ6WRATILJ2HKMRA', 'AE22KTPGG4CQPJEYMHDTAJDW76YQ', 'AE22TH7QS3VGRPLIT4CYEHAO

In [11]:
# ==== Build Strict Pre-Target and Training-Safe Histories ====
selected_key = selected_targets[
    ["user_id", "target_parent_asin", "target_timestamp_ms", "target_rank_desc"]
].copy()
history_join = reviews.merge(selected_key, on="user_id", how="inner")
prior_history = history_join[
    history_join["review_timestamp_ms"] < history_join["target_timestamp_ms"]
].copy()
prior_history = prior_history[
    prior_history["parent_asin"].astype(str).map(normalize_space)
    != prior_history["target_parent_asin"].astype(str).map(normalize_space)
].copy()
prior_history = prior_history.sort_values(
    ["user_id", "target_timestamp_ms", "review_timestamp_ms", "parent_asin"],
    ascending=[True, True, False, True],
    kind="mergesort",
)
prior_history["prior_rank_from_target"] = (
    prior_history.groupby(["user_id", "target_timestamp_ms"]).cumcount() + 1
)

observed_pre_target_counts = (
    prior_history.groupby(["user_id", "target_parent_asin", "target_timestamp_ms"])
    .agg(
        observed_pre_target_interactions=("parent_asin", "size"),
        observed_pre_target_unique_items=("parent_asin", "nunique"),
    )
    .reset_index()
)
selected_targets = selected_targets.merge(
    observed_pre_target_counts,
    on=["user_id", "target_parent_asin", "target_timestamp_ms"],
    how="left",
    validate="one_to_one",
)
selected_targets[["observed_pre_target_interactions", "observed_pre_target_unique_items"]] = (
    selected_targets[["observed_pre_target_interactions", "observed_pre_target_unique_items"]]
    .fillna(0)
    .astype(int)
)
if selected_targets["observed_pre_target_interactions"].ne(
    selected_targets["n_pre_target_interactions"]
).any() or selected_targets["observed_pre_target_unique_items"].ne(
    selected_targets["n_pre_target_unique_items"]
).any():
    mismatch = selected_targets[
        selected_targets["observed_pre_target_interactions"].ne(
            selected_targets["n_pre_target_interactions"]
        )
        | selected_targets["observed_pre_target_unique_items"].ne(
            selected_targets["n_pre_target_unique_items"]
        )
    ].head(20)
    display(mismatch)
    raise RuntimeError("Selected-target strict pre-target counts do not match rebuilt prior history.")

window_reviews_by_user = {
    user_id: group.copy()
    for user_id, group in window_reviews.groupby("user_id", sort=False)
}
discarded_counts = []
for _, row in selected_targets.iterrows():
    user_window = window_reviews_by_user.get(row["user_id"], window_reviews.iloc[0:0])
    discarded_counts.append(int((user_window["review_timestamp_ms"] > row["target_timestamp_ms"]).sum()))
selected_targets["discarded_newer_review_count"] = discarded_counts

prior_history_export = pd.DataFrame({
    "user_id": prior_history["user_id"],
    "target_parent_asin": prior_history["target_parent_asin"],
    "target_timestamp_ms": prior_history["target_timestamp_ms"].astype("int64"),
    "prior_parent_asin": prior_history["parent_asin"],
    "prior_timestamp_ms": prior_history["review_timestamp_ms"].astype("int64"),
    "prior_review_datetime": prior_history["review_datetime"],
    "prior_rank_from_target": prior_history["prior_rank_from_target"].astype(int),
    "prior_verified_purchase": prior_history["verified_purchase"],
})
print("Prior history rows:", len(prior_history_export))

Prior history rows: 18859


In [12]:
# ==== Define Herbal History Regimes ====

In [13]:
# ==== Assign Cold, Weak, and Strong Regimes ====
selected_targets["regime"] = selected_targets["n_pre_target_interactions"].map(regime_from_prior_count)
if set(selected_targets["regime"].dropna().unique()) - set(REGIME_ORDER):
    raise RuntimeError("Unexpected regime labels.")
if not selected_targets.loc[
    selected_targets["regime"].eq("strong"), "n_pre_target_interactions"
].ge(5).all():
    raise RuntimeError("Strong regime must remain n_pre_target_interactions >= 5.")
if not selected_targets.loc[
    selected_targets["regime"].eq("weak"), "n_pre_target_interactions"
].between(1, 4).all():
    raise RuntimeError("Weak regime must remain 1 <= n_pre_target_interactions <= 4.")
if not selected_targets.loc[
    selected_targets["regime"].eq("cold"), "n_pre_target_interactions"
].eq(0).all():
    raise RuntimeError("Cold regime must remain n_pre_target_interactions == 0.")
print(selected_targets["regime"].value_counts().reindex(REGIME_ORDER, fill_value=0).to_string())

regime
cold      24590
weak       3703
strong      670


In [14]:
# ==== Construct the Eligible Reserve and Fixed Within-Regime Order ====
# Retain every one-user target that passes the temporal, target-novelty,
# regime, and training-safe-history checks. Final balancing is deferred
# until query evidence has been audited downstream.
selected_targets = selected_targets.reset_index(drop=True).copy()
training_cutoff_ms = int(TRAIN_REVIEW_CUTOFF_EXCLUSIVE.value // 1_000_000)

available_counts = (
    selected_targets["regime"]
    .value_counts()
    .reindex(REGIME_ORDER, fill_value=0)
    .astype(int)
    .to_dict()
)
if any(int(available_counts[regime]) <= 0 for regime in REGIME_ORDER):
    raise RuntimeError(f"At least one regime has no eligible reserve cases: {available_counts}")

EXPECTED_ELIGIBLE_POOL_REGIME_COUNTS = dict(available_counts)
EXPECTED_ELIGIBLE_POOL_TOTAL_N = int(len(selected_targets))
SAMPLE_N_BY_REGIME = {}
EXPECTED_FINAL_TOTAL_N = None
DOWNSTREAM_QUERY_BALANCE_N_PER_REGIME = None
DOWNSTREAM_BALANCED_QUERY_REGIME_COUNTS = None
DOWNSTREAM_BALANCED_QUERY_TOTAL_N = None
limiting_regime = None
resolved_target_per_regime = None

# Assign a deterministic within-regime order so downstream balancing
# and same-regime replacement remain reproducible.
ordered_parts = []
for regime_index, regime in enumerate(REGIME_ORDER):
    group = selected_targets[selected_targets["regime"].eq(regime)].copy()
    ordered = (
        group.sample(frac=1.0, random_state=RANDOM_SEED, replace=False)
        .reset_index(drop=True)
    )
    ordered["selection_rank_within_regime"] = np.arange(1, len(ordered) + 1, dtype=int)
    ordered["initial_order_within_regime"] = ordered["selection_rank_within_regime"]
    ordered["same_regime_replacement_order"] = ordered["selection_rank_within_regime"]
    ordered["regime_order"] = int(regime_index)
    ordered_parts.append(ordered)

sampled_query_cases = (
    pd.concat(ordered_parts, ignore_index=True)
    .sort_values(
        ["regime_order", "selection_rank_within_regime"],
        kind="mergesort",
    )
    .reset_index(drop=True)
)
sampled_query_cases["initial_selected"] = pd.Series(
    pd.NA,
    index=sampled_query_cases.index,
    dtype="boolean",
)
sampled_query_cases["selection_stage"] = "deferred_to_query_audit"
sampled_counts = dict(available_counts)

if len(sampled_query_cases) != len(selected_targets):
    raise RuntimeError(
        "Notebook 03 discarded eligible reserve cases before query audit: "
        f"export={len(sampled_query_cases)}, eligible={len(selected_targets)}."
    )
if set(sampled_query_cases["review_row_id"].astype(int)) != set(selected_targets["review_row_id"].astype(int)):
    raise RuntimeError("Full eligible-reserve identity was not preserved.")
if sampled_query_cases["user_id"].duplicated().any():
    raise RuntimeError("Eligible reserve must contain at most one target per user.")

for regime in REGIME_ORDER:
    regime_ranks = (
        sampled_query_cases.loc[
            sampled_query_cases["regime"].eq(regime),
            "selection_rank_within_regime",
        ]
        .sort_values()
        .tolist()
    )
    expected_ranks = list(range(1, len(regime_ranks) + 1))
    if regime_ranks != expected_ranks:
        raise RuntimeError(f"Non-contiguous eligible-reserve order for regime {regime}.")

sampled_query_cases["case_id"] = [
    f"herbal_case_{idx:06d}"
    for idx in range(1, len(sampled_query_cases) + 1)
]
if sampled_query_cases["case_id"].duplicated().any():
    raise RuntimeError("Eligible reserve contains duplicate case_id values.")

final_selected_targets_by_rank = (
    sampled_query_cases["target_rank_desc"]
    .value_counts()
    .reindex(range(1, MAX_TARGET_RANK_ALLOWED + 1), fill_value=0)
    .astype(int)
    .to_dict()
)

print("Full eligible reserve counts:", available_counts)
print("Full eligible reserve total:", EXPECTED_ELIGIBLE_POOL_TOTAL_N)
print("Final regime quota: deferred to query audit")

# Build H_pre and H_train for the complete eligible reserve.
eligible_user_targets = sampled_query_cases[
    ["case_id", "user_id", "target_parent_asin", "target_timestamp_ms"]
].drop_duplicates()
history_join_eligible = reviews.merge(eligible_user_targets, on="user_id", how="inner")
prior_history_sampled_source = history_join_eligible[
    history_join_eligible["review_timestamp_ms"]
    < history_join_eligible["target_timestamp_ms"]
].copy()
prior_history_sampled_source = prior_history_sampled_source[
    prior_history_sampled_source["parent_asin"].astype(str).map(normalize_space)
    != prior_history_sampled_source["target_parent_asin"].astype(str).map(normalize_space)
].copy()
prior_history_sampled_source = prior_history_sampled_source.sort_values(
    ["case_id", "review_timestamp_ms", "parent_asin"],
    ascending=[True, False, True],
    kind="mergesort",
)
prior_history_sampled_source["prior_rank_from_target"] = (
    prior_history_sampled_source.groupby("case_id").cumcount() + 1
)
prior_history_sampled = pd.DataFrame({
    "case_id": prior_history_sampled_source["case_id"],
    "user_id": prior_history_sampled_source["user_id"],
    "target_parent_asin": prior_history_sampled_source["target_parent_asin"],
    "target_timestamp_ms": prior_history_sampled_source["target_timestamp_ms"].astype("int64"),
    "prior_parent_asin": prior_history_sampled_source["parent_asin"],
    "prior_item_id": prior_history_sampled_source["parent_asin"],
    "prior_timestamp_ms": prior_history_sampled_source["review_timestamp_ms"].astype("int64"),
    "prior_review_datetime": prior_history_sampled_source["review_datetime"],
    "prior_rank_from_target": prior_history_sampled_source["prior_rank_from_target"].astype(int),
    "prior_verified_purchase": prior_history_sampled_source["verified_purchase"],
})
training_prior_history_sampled = prior_history_sampled[
    prior_history_sampled["prior_timestamp_ms"]
    < np.minimum(prior_history_sampled["target_timestamp_ms"], training_cutoff_ms)
].copy()

eligible_reserve_case_ids = set(sampled_query_cases["case_id"].astype(str))
eligible_reserve_non_cold_case_ids = set(
    sampled_query_cases.loc[
        sampled_query_cases["regime"].ne("cold"), "case_id"
    ].astype(str)
)
strict_history_case_ids = set(prior_history_sampled["case_id"].astype(str))
training_history_case_ids = set(training_prior_history_sampled["case_id"].astype(str))

if not strict_history_case_ids.issubset(eligible_reserve_case_ids):
    raise RuntimeError("Strict prior history contains cases outside the full eligible reserve.")
if not training_history_case_ids.issubset(eligible_reserve_case_ids):
    raise RuntimeError("Training-safe prior history contains cases outside the full eligible reserve.")
if strict_history_case_ids != eligible_reserve_non_cold_case_ids:
    missing = sorted(eligible_reserve_non_cold_case_ids - strict_history_case_ids)
    extra = sorted(strict_history_case_ids - eligible_reserve_non_cold_case_ids)
    raise RuntimeError(
        "Strict prior-history export does not cover exactly the non-cold eligible reserve: "
        f"missing={len(missing)}, extra={len(extra)}, "
        f"missing_sample={missing[:10]}, extra_sample={extra[:10]}"
    )
if training_history_case_ids != eligible_reserve_non_cold_case_ids:
    missing = sorted(eligible_reserve_non_cold_case_ids - training_history_case_ids)
    extra = sorted(training_history_case_ids - eligible_reserve_non_cold_case_ids)
    raise RuntimeError(
        "Training-safe history export does not cover exactly the non-cold eligible reserve: "
        f"missing={len(missing)}, extra={len(extra)}, "
        f"missing_sample={missing[:10]}, extra_sample={extra[:10]}"
    )

history_export_case_coverage_qc = {
    "eligible_reserve_cases": int(len(eligible_reserve_case_ids)),
    "eligible_reserve_non_cold_cases": int(len(eligible_reserve_non_cold_case_ids)),
    "strict_history_cases": int(len(strict_history_case_ids)),
    "training_history_cases": int(len(training_history_case_ids)),
    "strict_history_matches_non_cold_eligible_reserve": True,
    "training_history_matches_non_cold_eligible_reserve": True,
}

sampled_pre_target_counts = (
    prior_history_sampled.groupby("case_id", dropna=False)
    .agg(
        observed_pre_target_interactions=("prior_parent_asin", "size"),
        observed_pre_target_unique_items=("prior_parent_asin", "nunique"),
    )
    .reset_index()
)
sampled_training_prior_counts = (
    training_prior_history_sampled.groupby("case_id", dropna=False)
    .agg(
        observed_train_safe_interactions=("prior_parent_asin", "size"),
        observed_train_safe_unique_items=("prior_parent_asin", "nunique"),
    )
    .reset_index()
)
sampled_prior_audit = sampled_query_cases[
    [
        "case_id",
        "user_id",
        "target_parent_asin",
        "target_timestamp_ms",
        "regime",
        "prior_history_n",
        "prior_item_n",
        "n_pre_target_interactions",
        "n_pre_target_unique_items",
        "n_train_safe_interactions",
        "n_train_safe_unique_items",
        "stage1_profile_available",
        "stage2_profile_available",
    ]
].merge(
    sampled_pre_target_counts,
    on="case_id",
    how="left",
    validate="one_to_one",
).merge(
    sampled_training_prior_counts,
    on="case_id",
    how="left",
    validate="one_to_one",
)
observed_columns = [
    "observed_pre_target_interactions",
    "observed_pre_target_unique_items",
    "observed_train_safe_interactions",
    "observed_train_safe_unique_items",
]
sampled_prior_audit[observed_columns] = sampled_prior_audit[observed_columns].fillna(0).astype(int)

history_mismatch = (
    sampled_prior_audit["observed_pre_target_interactions"].ne(
        sampled_prior_audit["n_pre_target_interactions"]
    )
    | sampled_prior_audit["observed_pre_target_unique_items"].ne(
        sampled_prior_audit["n_pre_target_unique_items"]
    )
    | sampled_prior_audit["observed_train_safe_interactions"].ne(
        sampled_prior_audit["n_train_safe_interactions"]
    )
    | sampled_prior_audit["observed_train_safe_unique_items"].ne(
        sampled_prior_audit["n_train_safe_unique_items"]
    )
)
if history_mismatch.any():
    mismatch = sampled_prior_audit.loc[history_mismatch].head(10)
    raise RuntimeError(
        "Eligible-reserve pre-target or training-safe history counts do not match: "
        f"{mismatch.to_dict(orient='records')}"
    )

if not sampled_query_cases["prior_history_n"].eq(
    sampled_query_cases["n_pre_target_interactions"]
).all():
    raise RuntimeError("prior_history_n must equal n_pre_target_interactions.")
if not sampled_query_cases["prior_item_n"].eq(
    sampled_query_cases["n_pre_target_unique_items"]
).all():
    raise RuntimeError("prior_item_n must equal n_pre_target_unique_items.")
if not sampled_query_cases["regime"].eq(
    sampled_query_cases["n_pre_target_interactions"].map(regime_from_prior_count)
).all():
    raise RuntimeError("Eligible-reserve regime labels must match strict pre-target history.")

bad_non_cold_training_prior = sampled_prior_audit[
    sampled_prior_audit["regime"].astype(str).str.lower().ne("cold")
    & sampled_prior_audit["observed_train_safe_unique_items"].lt(1)
]
print("Non-cold eligible-reserve cases with zero training-safe prior items:", len(bad_non_cold_training_prior))
if len(bad_non_cold_training_prior):
    display(
        bad_non_cold_training_prior[
            ["case_id", "regime", "observed_pre_target_interactions", "observed_train_safe_interactions"]
        ].sort_values(["regime", "case_id"])
    )
    raise RuntimeError("Every non-cold eligible-reserve case must retain training-safe prior history.")

bad_cold_history = sampled_prior_audit[
    sampled_prior_audit["regime"].astype(str).str.lower().eq("cold")
    & (
        sampled_prior_audit["observed_pre_target_interactions"].ne(0)
        | sampled_prior_audit["observed_train_safe_interactions"].ne(0)
    )
]
if len(bad_cold_history):
    raise RuntimeError("Cold cases must have zero strict pre-target and training-safe history.")

print("Eligible reserve counts:", sampled_counts)
print("Eligible reserve cases:", len(sampled_query_cases))
print("Full-reserve strict prior-history rows:", len(prior_history_sampled))
print("Full-reserve training-prior-history rows:", len(training_prior_history_sampled))


Full eligible reserve counts: {'cold': 24590, 'weak': 3703, 'strong': 670}
Full eligible reserve total: 28963
Final regime quota: deferred to query audit
Non-cold eligible-reserve cases with zero training-safe prior items: 0
Eligible reserve counts: {'cold': 24590, 'weak': 3703, 'strong': 670}
Eligible reserve cases: 28963
Full-reserve strict prior-history rows: 18859
Full-reserve training-prior-history rows: 14421


In [15]:
# ==== Export Eligible Cases and History Tables ====
metadata_cols = [
    'item_title', 'brand_primary', 'brand_norm', 'brand', 'manufacturer', 'sub_category', 'sub_category_norm',
    'item_form', 'item_form_norm', 'product_benefits', 'active_ingredients', 'special_ingredients',
    'primary_supplement_type', 'diet_type', 'material_feature', 'flavor', 'scent', 'family_brand',
    'family_sub_category', 'family_item_form', 'family_benefit_function', 'family_usage_need',
    'family_ingredient_composition', 'family_claims_diet', 'family_flavor', 'family_usage_target',
    'unit_count', 'number_of_items', 'package_dimensions', 'item_model_number', 'is_discontinued',
    'is_discontinued_norm',
]
base_case_cols = [
    'case_id', 'user_id', 'review_row_id', 'target_review_row_id', 'parent_asin', 'target_parent_asin', 'target_review_text', 'query_safe_text',
    'target_timestamp_ms', 'target_review_datetime', 'target_rank_desc', 'target_selection_mode',
    'selection_rank_within_regime', 'initial_order_within_regime',
    'same_regime_replacement_order', 'initial_selected', 'selection_stage',
    'discarded_newer_review_count', 'prior_history_n', 'prior_item_n',
    'n_pre_target_interactions', 'n_pre_target_unique_items',
    'n_train_safe_interactions', 'n_train_safe_unique_items',
    'stage1_profile_available', 'stage2_profile_available',
    'prior_same_item_review_n',
    'target_item_repeat_prior_flag', 'regime',
    'query_convertible_flag', 'query_convertibility_failure_reason', 'query_safe_token_count',
    'query_safe_signal_family_count', 'query_safe_signal_total_count',
    'c_lite_query_safe_signal_family_count', 'c_lite_query_safe_signal_total_count',
    'c_lite_token_fallback_pass', 'c_lite_signal_pass', 'direct_item_cue_hit_count',
]
sampled_query_cases['target_review_text'] = sampled_query_cases['review_text']
available_cols = list(dict.fromkeys(base_case_cols + [col for col in metadata_cols if col in sampled_query_cases.columns]))
sampled_query_cases_export = sampled_query_cases[available_cols].copy()
duplicate_export_cols = sampled_query_cases_export.columns[sampled_query_cases_export.columns.duplicated()].tolist()
if duplicate_export_cols:
    raise RuntimeError(f'Duplicate sampled_query_cases_export columns before export: {duplicate_export_cols}')

print('Prepared full eligible-reserve export rows:', len(sampled_query_cases_export))
print('Prepared prior history export rows:', len(prior_history_sampled))


Prepared full eligible-reserve export rows: 28963
Prepared prior history export rows: 18859


In [16]:
# ==== Assemble the Sampling Manifest ====
same_target_prior_leak_count = int(
    prior_history_sampled['prior_parent_asin'].astype(str).map(normalize_space).eq(
        prior_history_sampled['target_parent_asin'].astype(str).map(normalize_space)
    ).sum()
) if len(prior_history_sampled) else 0
future_or_same_time_prior_leak_count = int(
    pd.to_numeric(prior_history_sampled['prior_timestamp_ms'], errors='coerce').ge(
        pd.to_numeric(prior_history_sampled['target_timestamp_ms'], errors='coerce')
    ).sum()
) if len(prior_history_sampled) else 0
sampled_case_id_keys = sampled_query_cases_export[['case_id']].drop_duplicates()
prior_case_id_keys = prior_history_sampled[['case_id']].drop_duplicates() if len(prior_history_sampled) else pd.DataFrame({'case_id': []})
orphan_prior_history_row_count = int(
    prior_history_sampled.merge(sampled_case_id_keys, on='case_id', how='left', indicator=True)['_merge'].ne('both').sum()
) if len(prior_history_sampled) else 0
final_sample_counts = sampled_query_cases_export['regime'].value_counts().reindex(REGIME_ORDER, fill_value=0).astype(int).to_dict()
prior_history_count_matches_sampled_cases = bool(
    sampled_prior_audit['observed_pre_target_interactions'].eq(
        sampled_prior_audit['n_pre_target_interactions'].astype(int)
    ).all()
    and sampled_prior_audit['observed_pre_target_unique_items'].eq(
        sampled_prior_audit['n_pre_target_unique_items'].astype(int)
    ).all()
    and sampled_prior_audit['observed_train_safe_interactions'].eq(
        sampled_prior_audit['n_train_safe_interactions'].astype(int)
    ).all()
    and sampled_prior_audit['observed_train_safe_unique_items'].eq(
        sampled_prior_audit['n_train_safe_unique_items'].astype(int)
    ).all()
)

repeat_target_candidate_n = int(target_candidates['target_item_repeat_prior_flag'].sum())
repeat_target_candidate_user_n = int(
    target_candidates.loc[target_candidates['target_item_repeat_prior_flag'].eq(1), 'user_id'].nunique()
)
sampled_repeat_target_n = int(sampled_query_cases_export['target_item_repeat_prior_flag'].fillna(0).astype(int).sum()) if 'target_item_repeat_prior_flag' in sampled_query_cases_export.columns else 0

summary = {
    'stage': STAGE,
    'category_id': CATEGORY_ID,
    'category_folder': CATEGORY_FOLDER,
    'category_label': CATEGORY_LABEL,
    'artifact_schema_version': SAMPLING_ARTIFACT_SCHEMA_VERSION,
    'project_root': str(PROJECT_ROOT),
    'input_paths': {
        'reviews': str(REVIEWS_PATH),
        'item_schema': str(ITEM_SCHEMA_PATH),
            },
    'output_paths': {
        'sampled_query_cases_parquet': str(SAMPLED_QUERY_CASES_PARQUET),
        'sampled_query_cases_csv': str(SAMPLED_QUERY_CASES_CSV),
        'prior_history_parquet': str(PRIOR_HISTORY_PARQUET),
        'summary_json': str(OUTPUT_DIR / 'herbal_user_regime_sampling_summary.json'),
        'sampling_manifest_json': str(OUTPUT_DIR / 'herbal_user_regime_sampling_manifest.json'),
        'sampled_query_cases_compat_parquet': str(SAMPLED_QUERY_CASES_COMPAT_PARQUET),
        'sampled_query_cases_compat_csv': str(SAMPLED_QUERY_CASES_COMPAT_CSV),
        'prior_history_compat_parquet': str(PRIOR_HISTORY_COMPAT_PARQUET),
        'training_prior_history_parquet': str(TRAIN_PRIOR_HISTORY_PARQUET),
        'target_case_metadata_parquet': str(TARGET_CASE_METADATA_PARQUET),
        'summary_compat_json': str(PROCESSED_SAMPLING_DIR / 'herbal_user_regime_sampling_manifest.json'),
    },
    'evaluation_window_months': int(EVALUATION_WINDOW_MONTHS),
    'evaluation_window_start': EVALUATION_WINDOW_START.isoformat(),
    'evaluation_window_end': EVALUATION_WINDOW_END.isoformat(),
    'evaluation_window_start_inclusive': False,
    'evaluation_window_end_inclusive': True,
    'evaluation_window_filter_rule': 'review_datetime > EVALUATION_WINDOW_START and review_datetime <= EVALUATION_WINDOW_END',
    'target_selection_mode': TARGET_SELECTION_MODE,
    'target_rank_convention': TARGET_RANK_CONVENTION,
    'target_prior_same_item_policy': TARGET_PRIOR_SAME_ITEM_POLICY,
    'repeat_target_candidate_n': int(repeat_target_candidate_n),
    'repeat_target_candidate_user_n': int(repeat_target_candidate_user_n),
    'sampled_repeat_target_n': int(sampled_repeat_target_n),
    'sampling_decision_source': SAMPLING_DECISION_SOURCE,
    'sampling_decision_note': SAMPLING_DECISION_NOTE,
    'max_target_rank_allowed': int(MAX_TARGET_RANK_ALLOWED),
    'eligible_supply_by_regime_before_user_dedup': eligible_supply_by_regime_before_user_dedup,
    'eligible_candidate_rows_by_regime_before_user_dedup': eligible_candidate_rows_by_regime_before_user_dedup,
    'eligible_supply_by_regime_after_user_dedup': eligible_supply_by_regime_after_user_dedup,
    'users_with_no_eligible_review_in_top5_n': int(len(users_with_no_eligible_review_in_top5)),
    'selected_targets_by_rank': selected_targets_by_rank,
    'limiting_regime': None,
    'resolved_per_regime_sample_count': None,
    'sample_n_by_regime_requested': REQUESTED_SAMPLE_N_BY_REGIME,
    'sample_n_by_regime_resolved': {},
    'sample_n_by_regime_actual': final_sample_counts,
    'final_sample_counts': None,
    'final_total_n': None,
    'notebook_03_output_role': 'full_eligible_reserve_before_query_audit_and_balance',
    'eligible_pool_regime_counts_before_query_balance': final_sample_counts,
    'eligible_pool_total_n_before_query_balance': int(len(sampled_query_cases_export)),
    'downstream_query_balance_n_per_regime': None,
    'downstream_balanced_query_regime_counts': None,
    'downstream_balanced_query_total_n': None,
    'downstream_query_balancing_basis': DOWNSTREAM_QUERY_BALANCING_BASIS,
    'query_balance_deferred_to_query_audit': True,
    'initial_order_field': 'initial_order_within_regime',
    'same_regime_replacement_order_field': 'same_regime_replacement_order',
    'selection_rank_field': 'selection_rank_within_regime',
    'c_lite_requested_final_sample_counts': REQUESTED_SAMPLE_N_BY_REGIME,
    'c_lite_resolved_final_sample_counts': None,
    'sample_design_revision_reason': 'Notebook 03 exports every temporally and history-eligible target; query audit determines the final balanced quota and same-regime replacements downstream.',
    'query_eligibility_families': C_LITE_ELIGIBILITY_FAMILIES,
    'min_target_review_tokens': int(MIN_TARGET_REVIEW_TOKENS),
    'min_query_safe_token_count': int(MIN_QUERY_SAFE_TOKEN_COUNT),
    'final_min_query_tokens': int(FINAL_MIN_QUERY_TOKENS),
    'final_max_query_tokens': int(FINAL_MAX_QUERY_TOKENS),
    'final_min_query_safe_signal_families': int(FINAL_MIN_QUERY_SAFE_SIGNAL_FAMILIES),
    'final_min_query_safe_signal_total': int(FINAL_MIN_QUERY_SAFE_SIGNAL_TOTAL),
    'min_signal_families': int(MIN_SIGNAL_FAMILIES),
    'min_signal_total': int(MIN_SIGNAL_TOTAL),
    'min_safe_token_fallback': int(MIN_SAFE_TOKEN_FALLBACK),
    'c_lite_signal_pass_rule': 'benefit/form/claim signal pass OR safe token fallback >= 3 after direct-cue scrubbing',
    'direct_cue_cols': DIRECT_CUE_COLS,
    'title_phrase_cue_cols': TITLE_PHRASE_CUE_COLS,
    'title_token_scrubbing_used': False,
    'generic_benefit_form_claim_scrubbing_used': False,
    'sampled_query_cases_n': int(len(sampled_query_cases_export)),
    'prior_history_rows_n': int(len(prior_history_sampled)),
    'regime_definition': REGIME_DEFINITION,
    'regime_history_scope': 'strict_pre_target_excluding_target_item',
    'stage1_profile_history_scope': 'strict_pre_target_and_before_training_cutoff_excluding_target_item',
    'stage2_profile_history_scope': 'strict_pre_target_excluding_target_item',
    'non_cold_stage1_profile_required': True,
    'target_candidate_history_recomputed_per_target': True,
    'category_specific_regime_design': CATEGORY_SPECIFIC_REGIME_DESIGN,
    'moderate_regime_used': MODERATE_REGIME_USED,
    'regime_design_note': 'Herbal uses cold=0, weak=1-4, strong>=5, without a moderate regime.',
    'regime_convention_note': 'Herbal uses cold=0, weak=1-4, strong>=5, with no moderate regime; this category-specific design reflects lower user-history density and is enforced downstream by notebooks 05/06.',
    'artifact_design_note': 'Herbal 02 writes canonical interim outputs and processed/user_sampling compatibility aliases while preserving the embedded final sampling design.',
    'leakage_rule': 'For rank > 1 targets, newer same-user reviews are discarded and excluded from prior history and downstream user evidence.',
    'prior_history_rule': 'Prior history uses only same-user reviews with review_timestamp_ms strictly earlier than target_timestamp_ms and parent_asin different from target_parent_asin.',
    'training_prior_history_rule': 'prior_timestamp_ms < min(target_timestamp_ms, training_review_cutoff_exclusive)',
    'training_review_cutoff_exclusive': TRAIN_REVIEW_CUTOFF_EXCLUSIVE.isoformat(),
    'training_prior_history_rows_n': int(len(training_prior_history_sampled)),
    'training_prior_history_cases_n': int(training_prior_history_sampled['case_id'].nunique()),
    'same_target_prior_items_excluded': True,
    'prior_history_recomputed_after_same_target_exclusion': True,
    'prior_history_case_universe_validated': True,
    'history_export_case_coverage_qc': history_export_case_coverage_qc,
    'strict_prior_history_case_scope': 'full_eligible_reserve_before_query_audit_and_balance',
    'training_prior_history_case_scope': 'full_eligible_reserve_before_query_audit_and_balance',
    'common_critical_sampling_contract': COMMON_CRITICAL_SAMPLING_CONTRACT,
    'common_critical_sampling_contract_sha256': COMMON_CRITICAL_SAMPLING_CONTRACT_SHA256,
    'prior_history_count_matches_sampled_cases': prior_history_count_matches_sampled_cases,
    'same_target_prior_leak_count': same_target_prior_leak_count,
    'future_or_same_time_prior_leak_count': future_or_same_time_prior_leak_count,
    'orphan_prior_history_row_count': orphan_prior_history_row_count,
    'rating_used': False,
    'sentiment_used': False,
    'rating_exported': False,
    'helpful_vote_exported': False,
    'raw_review_text_exported': False,
    'prior_review_text_exported': False,
    'verified_purchase_exported': True,
    'selected_target_traceability_columns': ['review_row_id', 'target_review_row_id'],
    'random_seed': RANDOM_SEED,
}
print('Prepared summary JSON payload.')


Prepared summary JSON payload.


In [17]:
# ==== Validate Temporal and Identity Contracts ====
eligible_reserve_counts = {
    regime: int(
        sampled_query_cases_export["regime"].eq(regime).sum()
    )
    for regime in REGIME_ORDER
}

if sampled_query_cases_export["user_id"].duplicated().any():
    raise RuntimeError("Eligible reserve has duplicate user_id values.")

if sampled_query_cases_export[["user_id", "target_parent_asin", "target_timestamp_ms"]].duplicated().any():
    raise RuntimeError("Eligible reserve has duplicate selected targets per user.")

if EXPECTED_ELIGIBLE_POOL_REGIME_COUNTS is not None and eligible_reserve_counts != EXPECTED_ELIGIBLE_POOL_REGIME_COUNTS:
    raise RuntimeError(
        f"Eligible-reserve counts mismatch: actual {eligible_reserve_counts}, "
        f"expected {EXPECTED_ELIGIBLE_POOL_REGIME_COUNTS}."
    )

if EXPECTED_ELIGIBLE_POOL_TOTAL_N is not None and len(sampled_query_cases_export) != EXPECTED_ELIGIBLE_POOL_TOTAL_N:
    raise RuntimeError(
        f"Eligible-reserve total mismatch: actual {len(sampled_query_cases_export)}, "
        f"expected {EXPECTED_ELIGIBLE_POOL_TOTAL_N}."
    )

required_order_columns = [
    "selection_rank_within_regime",
    "initial_order_within_regime",
    "same_regime_replacement_order",
    "initial_selected",
    "selection_stage",
]
missing_order_columns = [
    column for column in required_order_columns
    if column not in sampled_query_cases_export.columns
]
if missing_order_columns:
    raise RuntimeError(f"Eligible reserve is missing downstream-order columns: {missing_order_columns}")
if sampled_query_cases_export["initial_selected"].notna().any():
    raise RuntimeError("Notebook 03 must not select final query slots before query audit.")
if not sampled_query_cases_export["selection_stage"].eq("deferred_to_query_audit").all():
    raise RuntimeError("Eligible-reserve selection stage must remain deferred to query audit.")
for regime in REGIME_ORDER:
    regime_ranks = (
        sampled_query_cases_export.loc[
            sampled_query_cases_export["regime"].eq(regime),
            "selection_rank_within_regime",
        ]
        .sort_values()
        .tolist()
    )
    if regime_ranks != list(range(1, len(regime_ranks) + 1)):
        raise RuntimeError(f"Eligible-reserve order is not contiguous for regime {regime}.")

if not sampled_query_cases_export["target_rank_desc"].between(1, MAX_TARGET_RANK_ALLOWED).all():
    raise RuntimeError(f"All eligible-reserve target_rank_desc values must be between 1 and {MAX_TARGET_RANK_ALLOWED}.")

if not sampled_query_cases_export["target_selection_mode"].eq(TARGET_SELECTION_MODE).all():
    raise RuntimeError(f"All eligible-reserve cases must use target_selection_mode = {TARGET_SELECTION_MODE}.")

if "target_item_repeat_prior_flag" in sampled_query_cases_export.columns:
    if sampled_query_cases_export["target_item_repeat_prior_flag"].fillna(0).astype(int).sum() > 0:
        raise RuntimeError("Eligible reserve contains target items previously reviewed by the same user.")

if "prior_same_item_review_n" in sampled_query_cases_export.columns:
    if sampled_query_cases_export["prior_same_item_review_n"].fillna(0).astype(int).sum() > 0:
        raise RuntimeError("Eligible reserve contains targets with prior same-item reviews.")

if not sampled_query_cases_export["query_convertible_flag"].eq(1).all():
    raise RuntimeError("All eligible-reserve cases must be query-convertible.")

if sampled_query_cases_export["query_safe_text"].fillna("").astype(str).map(normalize_space).eq("").any():
    raise RuntimeError("All eligible-reserve cases must have non-empty query_safe_text.")

if set(sampled_query_cases_export["regime"].dropna().unique()) - set(REGIME_ORDER):
    raise RuntimeError("Eligible reserve contains unexpected regimes.")

observed_final_counts = (
    sampled_query_cases_export["regime"]
    .value_counts()
    .reindex(REGIME_ORDER, fill_value=0)
    .astype(int)
    .to_dict()
)

if EXPECTED_ELIGIBLE_POOL_REGIME_COUNTS is not None and observed_final_counts != EXPECTED_ELIGIBLE_POOL_REGIME_COUNTS:
    raise RuntimeError(
        f"Versioned eligible-pool counts mismatch: actual {observed_final_counts}, "
        f"expected {EXPECTED_ELIGIBLE_POOL_REGIME_COUNTS}."
    )

if not sampled_query_cases_export.loc[sampled_query_cases_export["regime"].eq("strong"), "n_pre_target_interactions"].ge(5).all():
    raise RuntimeError("Strong rows must have n_pre_target_interactions >= 5.")

if not sampled_query_cases_export.loc[sampled_query_cases_export["regime"].eq("weak"), "n_pre_target_interactions"].between(1, 4).all():
    raise RuntimeError("Weak rows must have 1 <= n_pre_target_interactions <= 4.")

if not sampled_query_cases_export.loc[sampled_query_cases_export["regime"].eq("cold"), "n_pre_target_interactions"].eq(0).all():
    raise RuntimeError("Cold rows must have n_pre_target_interactions == 0.")

if not sampled_query_cases_export["prior_history_n"].eq(sampled_query_cases_export["n_pre_target_interactions"]).all():
    raise RuntimeError("prior_history_n must equal n_pre_target_interactions.")
if not sampled_query_cases_export["prior_item_n"].eq(sampled_query_cases_export["n_pre_target_unique_items"]).all():
    raise RuntimeError("prior_item_n must equal n_pre_target_unique_items.")

if len(prior_history_sampled) and not (
    prior_history_sampled["prior_timestamp_ms"] < prior_history_sampled["target_timestamp_ms"]
).all():
    raise RuntimeError("Prior history must use timestamps strictly earlier than target timestamp.")

if len(prior_history_sampled) and prior_history_sampled["prior_parent_asin"].astype(str).map(normalize_space).eq(
    prior_history_sampled["target_parent_asin"].astype(str).map(normalize_space)
).any():
    raise RuntimeError("Prior history must not include the selected target item.")

forbidden_prior_export_cols = {
    "prior_review_text",
    "review_text",
    "raw_review_text",
    "target_review_text",
    "rating",
    "helpful_vote",
    "sentiment",
}

forbidden_prior_export_present = sorted(forbidden_prior_export_cols & set(prior_history_sampled.columns))

if forbidden_prior_export_present:
    raise RuntimeError(
        f"Prior history export contains forbidden text/rating columns: {forbidden_prior_export_present}"
    )

if "case_id" in prior_history_sampled.columns:
    orphan_prior_rows = prior_history_sampled.merge(
        sampled_query_cases_export[["case_id"]],
        on="case_id",
        how="left",
        indicator=True,
    )
else:
    orphan_prior_rows = prior_history_sampled.merge(
        sampled_query_cases_export[["user_id", "target_parent_asin", "target_timestamp_ms"]],
        on=["user_id", "target_parent_asin", "target_timestamp_ms"],
        how="left",
        indicator=True,
    )

if len(orphan_prior_rows) and orphan_prior_rows["_merge"].ne("both").any():
    raise RuntimeError("Prior history contains rows outside the eligible reserve.")

history_count_mismatch = (
    sampled_prior_audit["observed_pre_target_interactions"].ne(
        sampled_prior_audit["n_pre_target_interactions"].astype(int)
    )
    | sampled_prior_audit["observed_pre_target_unique_items"].ne(
        sampled_prior_audit["n_pre_target_unique_items"].astype(int)
    )
    | sampled_prior_audit["observed_train_safe_interactions"].ne(
        sampled_prior_audit["n_train_safe_interactions"].astype(int)
    )
    | sampled_prior_audit["observed_train_safe_unique_items"].ne(
        sampled_prior_audit["n_train_safe_unique_items"].astype(int)
    )
)

if history_count_mismatch.any():
    mismatch = sampled_prior_audit[history_count_mismatch].head(10)
    raise RuntimeError(
        "Eligible-reserve pre-target or training-safe history counts do not match: "
        f"{mismatch.to_dict(orient='records')}"
    )

eval_start_ms = int(EVALUATION_WINDOW_START.value // 1_000_000)
eval_end_ms = int(EVALUATION_WINDOW_END.value // 1_000_000)
training_cutoff_ms = int(TRAIN_REVIEW_CUTOFF_EXCLUSIVE.value // 1_000_000)

targets_outside_window = int(
    sampled_query_cases_export["target_timestamp_ms"].le(eval_start_ms).sum()
    + sampled_query_cases_export["target_timestamp_ms"].gt(eval_end_ms).sum()
)
strict_prior_rows_at_or_after_target = int(
    prior_history_sampled["prior_timestamp_ms"].ge(prior_history_sampled["target_timestamp_ms"]).sum()
) if len(prior_history_sampled) else 0
training_prior_rows_at_or_after_target = int(
    training_prior_history_sampled["prior_timestamp_ms"].ge(training_prior_history_sampled["target_timestamp_ms"]).sum()
) if len(training_prior_history_sampled) else 0
training_prior_rows_at_or_after_cutoff = int(
    training_prior_history_sampled["prior_timestamp_ms"].ge(training_cutoff_ms).sum()
) if len(training_prior_history_sampled) else 0
same_target_item_prior_rows = int(
    prior_history_sampled["prior_parent_asin"].astype(str).map(normalize_space).eq(
        prior_history_sampled["target_parent_asin"].astype(str).map(normalize_space)
    ).sum()
) if len(prior_history_sampled) else 0

recomputed_regime = sampled_query_cases_export["n_pre_target_interactions"].map(regime_from_prior_count)
regime_mismatch_mask = sampled_query_cases_export["regime"].ne(recomputed_regime)
regime_mismatch_cases = int(regime_mismatch_mask.sum())
non_cold_zero_strict_prior = sampled_query_cases_export[
    sampled_query_cases_export["regime"].astype(str).str.lower().ne("cold")
    & sampled_query_cases_export["n_pre_target_interactions"].eq(0)
]

history_subset_key_cols = ["case_id", "target_timestamp_ms", "prior_parent_asin", "prior_timestamp_ms"]
if len(training_prior_history_sampled):
    pre_key_counts = (
        prior_history_sampled.groupby(history_subset_key_cols, dropna=False)
        .size()
        .rename("pre_count")
        .reset_index()
    )
    train_key_counts = (
        training_prior_history_sampled.groupby(history_subset_key_cols, dropna=False)
        .size()
        .rename("train_count")
        .reset_index()
    )
    train_subset_check = train_key_counts.merge(pre_key_counts, on=history_subset_key_cols, how="left")
    train_subset_violations = int(
        train_subset_check["pre_count"].fillna(0).lt(train_subset_check["train_count"]).sum()
    )
else:
    train_subset_violations = 0

bad_non_cold_profile = sampled_prior_audit[
    sampled_prior_audit["regime"].ne("cold")
    & sampled_prior_audit["observed_train_safe_unique_items"].lt(1)
]
non_cold_zero_training_safe_prior = bad_non_cold_profile.copy()

temporal_validation_summary = {
    "evaluation_window_start": EVALUATION_WINDOW_START.isoformat(),
    "evaluation_window_end": EVALUATION_WINDOW_END.isoformat(),
    "selected_target_count": int(len(sampled_query_cases_export)),
    "target_rank_convention": TARGET_RANK_CONVENTION,
    "selected_targets_by_rank": final_selected_targets_by_rank,
    "users_with_no_eligible_review_in_top5": int(len(users_with_no_eligible_review_in_top5)),
    "eligible_supply_by_regime_before_user_dedup": eligible_supply_by_regime_before_user_dedup,
    "eligible_supply_by_regime_after_user_dedup": eligible_supply_by_regime_after_user_dedup,
    "limiting_regime": None,
    "resolved_per_regime_sample_count": None,
    "eligible_reserve_count_by_regime": observed_final_counts,
    "query_balance_deferred_to_query_audit": True,
    "targets_outside_window": targets_outside_window,
    "strict_prior_rows_at_or_after_target": strict_prior_rows_at_or_after_target,
    "training_prior_rows_at_or_after_target": training_prior_rows_at_or_after_target,
    "training_prior_rows_at_or_after_cutoff": training_prior_rows_at_or_after_cutoff,
    "same_target_item_prior_rows": same_target_item_prior_rows,
    "regime_mismatch_cases": regime_mismatch_cases,
    "h_train_not_subset_of_h_pre_rows": train_subset_violations,
    "non_cold_cases_with_zero_strict_prior": int(len(non_cold_zero_strict_prior)),
    "non_cold_cases_with_zero_training_safe_prior": int(len(non_cold_zero_training_safe_prior)),
}
print("Temporal target/history validation summary:")
for key, value in temporal_validation_summary.items():
    print(f"- {key}: {value}")
if len(non_cold_zero_training_safe_prior):
    print("Non-cold zero training-safe prior case identifiers:")
    display(non_cold_zero_training_safe_prior[["case_id", "regime", "observed_pre_target_interactions", "observed_train_safe_interactions"]].sort_values(["regime", "case_id"]))

if targets_outside_window:
    raise RuntimeError("Selected targets must lie inside the start-exclusive, end-inclusive evaluation window.")
if strict_prior_rows_at_or_after_target:
    raise RuntimeError("Strict prior history contains rows at or after the selected target timestamp.")
if training_prior_rows_at_or_after_target:
    raise RuntimeError("Training-safe prior history contains rows at or after the selected target timestamp.")
if training_prior_rows_at_or_after_cutoff:
    raise RuntimeError("Training-safe prior history contains rows at or after TRAIN_REVIEW_CUTOFF_EXCLUSIVE.")
if same_target_item_prior_rows:
    raise RuntimeError("Prior history contains the selected target parent_asin.")
if regime_mismatch_cases:
    raise RuntimeError("Stored regime does not match regime recomputed from effective strict-prior count.")
if train_subset_violations:
    raise RuntimeError("Training-safe history must be a subset of strict pre-target history.")
if len(non_cold_zero_strict_prior):
    raise RuntimeError("Non-cold eligible-reserve cases must have at least one strict pre-target interaction.")

if len(prior_history_sampled):
    prior_with_rank = prior_history_sampled.merge(
        sampled_query_cases_export[
            ["user_id", "target_parent_asin", "target_timestamp_ms", "target_rank_desc"]
        ],
        on=["user_id", "target_parent_asin", "target_timestamp_ms"],
        how="left",
    )

    non_latest_rank_prior = prior_with_rank[prior_with_rank["target_rank_desc"].gt(1)]

    if len(non_latest_rank_prior) and non_latest_rank_prior["prior_timestamp_ms"].ge(non_latest_rank_prior["target_timestamp_ms"]).any():
        raise RuntimeError(
            "For rank > 1 targets, prior history contains a timestamp greater than or equal to "
            "the selected target timestamp; discarded newer reviews must not be included."
        )

print(
    "Discarded newer review leakage validation passed: "
    "all prior_timestamp_ms values are strictly earlier than target_timestamp_ms."
)

if TRAIN_REVIEW_CUTOFF_EXCLUSIVE != EVALUATION_WINDOW_START:
    raise RuntimeError(
        "The historical-review cutoff must equal the evaluation-window start."
    )

if len(training_prior_history_sampled):
    train_cutoff_ms = int(TRAIN_REVIEW_CUTOFF_EXCLUSIVE.value // 1_000_000)
    effective_cutoff = np.minimum(
        training_prior_history_sampled["target_timestamp_ms"].to_numpy(),
        train_cutoff_ms,
    )
    if not (
        training_prior_history_sampled["prior_timestamp_ms"].to_numpy()
        < effective_cutoff
    ).all():
        raise RuntimeError("Training-period prior history violates the effective cutoff.")

print("Temporal target/history validation: passed")

rating_used = False
sentiment_used = False
rating_exported = False
helpful_vote_exported = False
verified_purchase_exported = True

if rating_used or sentiment_used:
    raise RuntimeError("Rating and sentiment must not be used for query convertibility or query generation.")

if rating_exported or helpful_vote_exported:
    raise RuntimeError("Rating and helpful-vote must not be exported.")

if not verified_purchase_exported:
    raise RuntimeError("Verified-purchase should remain exported for prior history.")

sampled_query_cases_export.to_parquet(SAMPLED_QUERY_CASES_PARQUET, index=False)
sampled_query_cases_export.to_csv(SAMPLED_QUERY_CASES_CSV, index=False, encoding="utf-8-sig")
prior_history_sampled.to_parquet(PRIOR_HISTORY_PARQUET, index=False)

sampled_query_cases_export.to_parquet(SAMPLED_QUERY_CASES_COMPAT_PARQUET, index=False)
sampled_query_cases_export.to_csv(SAMPLED_QUERY_CASES_COMPAT_CSV, index=False, encoding="utf-8-sig")
prior_history_sampled.to_parquet(PRIOR_HISTORY_COMPAT_PARQUET, index=False)
training_prior_history_sampled.to_parquet(TRAIN_PRIOR_HISTORY_PARQUET, index=False)

with open(OUTPUT_DIR / "herbal_user_regime_sampling_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

with open(OUTPUT_DIR / "herbal_user_regime_sampling_manifest.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

with open(PROCESSED_SAMPLING_DIR / "herbal_user_regime_sampling_manifest.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("Validation complete.")
print("Final QC summary:")
print("Eligible pool total before query balancing:", len(sampled_query_cases_export))
print("Downstream balanced query total: deferred to query audit")
print("Downstream balance quota per regime: deferred to query audit")
print("Eligible reserve size:", len(sampled_query_cases_export))
print("Eligible reserve regime counts:", observed_final_counts)
print("Max target rank:", int(sampled_query_cases_export["target_rank_desc"].max()))
print("Target selection mode:", TARGET_SELECTION_MODE)
print("Full eligible reserve:", SAMPLED_QUERY_CASES_PARQUET)
print("Prior history:", PRIOR_HISTORY_PARQUET)
print("Training prior history:", TRAIN_PRIOR_HISTORY_PARQUET)
print("Sampling manifest:", OUTPUT_DIR / "herbal_user_regime_sampling_manifest.json")


Temporal target/history validation summary:
- evaluation_window_start: 2022-03-31T23:56:10.358000+00:00
- evaluation_window_end: 2022-12-31T23:56:10.358000+00:00
- selected_target_count: 28963
- target_rank_convention: target_rank_desc is one-based within the evaluation window: 1=latest review, 2=second latest, ..., 5=fifth latest.
- selected_targets_by_rank: {1: 26868, 2: 1462, 3: 392, 4: 135, 5: 106}
- users_with_no_eligible_review_in_top5: 48652
- eligible_supply_by_regime_before_user_dedup: {'cold': 24590, 'weak': 3809, 'strong': 670}
- eligible_supply_by_regime_after_user_dedup: {'cold': 24590, 'weak': 3703, 'strong': 670}
- limiting_regime: None
- resolved_per_regime_sample_count: None
- eligible_reserve_count_by_regime: {'cold': 24590, 'weak': 3703, 'strong': 670}
- query_balance_deferred_to_query_audit: True
- targets_outside_window: 0
- strict_prior_rows_at_or_after_target: 0
- training_prior_rows_at_or_after_target: 0
- training_prior_rows_at_or_after_cutoff: 0
- same_target_

In [18]:
# ==== Export Target-Case Metadata ====
from pathlib import Path
import os

import pandas as pd

PROJECT_ROOT = Path("/content/drive/MyDrive/thesis_recsys/categories/herbal_supplements")
PROCESSED_SAMPLING_DIR = PROJECT_ROOT / "data" / "processed" / "user_sampling"

SAMPLED_QUERY_CASES_COMPAT_PARQUET = PROCESSED_SAMPLING_DIR / "herbal_user_regime_sample.parquet"
TARGET_CASE_METADATA_PARQUET = PROCESSED_SAMPLING_DIR / "herbal_target_case_metadata.parquet"

if "sampled_query_cases_export" in globals():
    final_sampling_pool = sampled_query_cases_export.copy()
else:
    final_sampling_pool = pd.read_parquet(SAMPLED_QUERY_CASES_COMPAT_PARQUET)

required_source_cols = [
    "case_id",
    "user_id",
    "target_parent_asin",
    "target_timestamp_ms",
    "regime",
]
optional_source_cols = [
    "target_review_datetime",
    "target_rank_desc",
    "target_selection_mode",
    "sampling_bracket",
    "initial_selected",
    "prior_history_n",
    "prior_item_n",
    "target_review_row_id",
    "review_row_id",
    "target_item_repeat_prior_flag",
    "n_pre_target_interactions",
    "n_pre_target_unique_items",
    "n_train_safe_interactions",
    "n_train_safe_unique_items",
    "stage1_profile_available",
    "stage2_profile_available",
]

missing_source_cols = [col for col in required_source_cols if col not in final_sampling_pool.columns]
if missing_source_cols:
    raise RuntimeError(f"Full Herbal sampling pool missing required target metadata columns: {missing_source_cols}")

metadata_cols = list(dict.fromkeys(required_source_cols + [col for col in optional_source_cols if col in final_sampling_pool.columns]))
target_case_metadata = final_sampling_pool[metadata_cols].copy()

if target_case_metadata["case_id"].isna().any():
    raise RuntimeError("target_case_metadata has null case_id.")

if target_case_metadata["case_id"].duplicated().any():
    raise RuntimeError("target_case_metadata has duplicate case_id rows.")

for col in ["user_id", "target_parent_asin", "target_timestamp_ms", "regime"]:
    if target_case_metadata[col].isna().any():
        raise RuntimeError(f"target_case_metadata has null values in {col}.")

target_case_metadata["target_timestamp_ms"] = pd.to_numeric(
    target_case_metadata["target_timestamp_ms"],
    errors="coerce",
)
if target_case_metadata["target_timestamp_ms"].isna().any():
    raise RuntimeError("target_case_metadata has null values in target_timestamp_ms.")
target_case_metadata["target_timestamp_ms"] = target_case_metadata["target_timestamp_ms"].astype("int64")

missing_target_metadata_cases = sorted(
    set(final_sampling_pool["case_id"].astype(str))
    - set(target_case_metadata["case_id"].astype(str))
)

if missing_target_metadata_cases:
    raise RuntimeError(
        "target_case_metadata does not cover the full sampling pool. "
        f"Missing case_id count: {len(missing_target_metadata_cases)}"
    )

if len(target_case_metadata) != len(final_sampling_pool):
    raise RuntimeError(
        "target_case_metadata row count does not match the full sampling pool: "
        f"metadata={len(target_case_metadata)}, pool={len(final_sampling_pool)}"
    )

TARGET_CASE_METADATA_PARQUET.parent.mkdir(parents=True, exist_ok=True)
target_case_metadata.to_parquet(TARGET_CASE_METADATA_PARQUET, index=False)

print("Saved target case metadata:", TARGET_CASE_METADATA_PARQUET)
print("Rows:", len(target_case_metadata))
print("Unique case_id:", target_case_metadata["case_id"].nunique())
print("Regime counts:", target_case_metadata["regime"].value_counts().reindex(REGIME_ORDER, fill_value=0).astype(int).to_dict() if "REGIME_ORDER" in globals() else target_case_metadata["regime"].value_counts().astype(int).to_dict())


Saved target case metadata: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/data/processed/user_sampling/herbal_target_case_metadata.parquet
Rows: 28963
Unique case_id: 28963
Regime counts: {'cold': 24590, 'weak': 3703, 'strong': 670}


In [19]:
# ==== Summarize Query-Eligibility Diagnostics ====
diagnostic_cols = [
    "query_convertibility_failure_reason",
    "query_safe_token_count",
    "query_safe_signal_family_count",
    "query_safe_signal_total_count",
    "c_lite_query_safe_signal_family_count",
    "c_lite_query_safe_signal_total_count",
    "target_rank_desc",
    "target_item_repeat_prior_flag",
]

available_diagnostic_cols = [col for col in diagnostic_cols if col in target_candidates.columns]

print("Target candidate diagnostics by convertibility:")
display(
    target_candidates
    .groupby("query_convertible_flag")[available_diagnostic_cols]
    .describe()
)

print("Failure reasons:")
display(
    target_candidates.loc[target_candidates["query_convertible_flag"].eq(0), "query_convertibility_failure_reason"]
    .value_counts()
    .head(20)
)

print("Convertible candidates by target rank:")
display(
    target_candidates
    .groupby(["target_rank_desc", "query_convertible_flag"])
    .size()
    .unstack(fill_value=0)
)

print("C-lite signal family counts:")
display(
    target_candidates[[
        "query_safe_signal_family_count",
        "query_safe_signal_total_count",
        "c_lite_query_safe_signal_family_count",
        "c_lite_query_safe_signal_total_count",
    ]].describe()
)


Target candidate diagnostics by convertibility:


query_safe_token_count                                                      query_safe_signal_family_count                                               \
                                        count       mean        std  min   25%   50%   75%     max                          count      mean       std  min  25%  50%  75%  max   
query_convertible_flag                                                                                                                                                           
0                                     55242.0  26.678849  28.574272  0.0   9.0  18.0  34.0   620.0                        55242.0  0.011567  0.124016  0.0  0.0  0.0  0.0  4.0   
1                                     35246.0  61.241049  62.458116  3.0  23.0  43.0  77.0  1487.0                        35246.0  1.357799  0.634891  1.0  1.0  1.0  2.0  5.0   

                       query_safe_signal_total_count                                               c_lite_query_safe_signal_family_count                                               \
                                               count      mean       std  min  25%  50%  75%   max                                 count      mean       std  min  25%  50%  75%  max   
query_convertible_flag                                                                                                                                                                  
0                                            55242.0  0.013341  0.153828  0.0  0.0  0.0  0.0   7.0                               55242.0  0.008544  0.098317  0.0  0.0  0.0  0.0  3.0   
1                                            35246.0  1.610027  1.020166  1.0  1.0  1.0  2.0  12.0                               35246.0  1.018754  0.552958  0.0  1.0  1.0  1.0  3.0   

                       c_lite_query_safe_signal_total_count                                               target_rank_desc                                              target_item_repeat_prior_flag  \
                                                      count      mean       std  min  25%  50%  75%   max            count      mean       std  min  25%  50%  75%  max                         count   
query_convertible_flag                                                                                                                                                                                  
0                                                   55242.0  0.010137  0.126095  0.0  0.0  0.0  0.0   5.0          55242.0  1.167264  0.534748  1.0  1.0  1.0  1.0  5.0                       55242.0   
1                                                   35246.0  1.222153  0.862953  0.0  1.0  1.0  2.0  10.0          35246.0  1.311525  0.783636  1.0  1.0  1.0  1.0  5.0                       35246.0   

                                                                     
                            mean       std  min  25%  50%  75%  max  
query_convertible_flag                                               
0                       0.001575  0.039654  0.0  0.0  0.0  0.0  1.0  
1                       0.002951  0.054241  0.0  0.0  0.0  0.0  1.0

Failure reasons:


,count
query_convertibility_failure_reason,
low_signal_family_count,49697
short_review,4875
discontinued_item,670


Convertible candidates by target rank:


query_convertible_flag,0,1
target_rank_desc,,
1,48780,28835
2,4705,3774
3,988,1246
4,517,850
5,252,541


C-lite signal family counts:


,query_safe_signal_family_count,query_safe_signal_total_count,c_lite_query_safe_signal_family_count,c_lite_query_safe_signal_total_count
count,90488.000000,90488.000000,90488.000000,90488.000000
mean,0.535938,0.635267,0.402031,0.482230
std,0.772889,1.012942,0.606359,0.805656
min,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000
75%,1.000000,1.000000,1.000000,1.000000
max,5.000000,12.000000,3.000000,10.000000


In [20]:
# ==== Export the Cross-Category Sampling Contract ====
def summarize_common_frame(frame_name, df):
    rows = []
    if df is None:
        return rows
    for common_name, col in COMMON_SAMPLING_COLUMNS.items():
        rows.append({
            "frame_name": frame_name,
            "common_column": common_name,
            "source_column": col,
            "present": bool(col in df.columns),
            "non_null_rows": int(df[col].notna().sum()) if col in df.columns else 0,
            "unique_values": int(df[col].nunique(dropna=True)) if col in df.columns else 0,
        })
    return rows

common_frame_candidates = {
    "sampled_query_cases": globals().get("sampled_query_cases_export", globals().get("sampled_query_cases_df", globals().get("sampled_users_df"))),
    "target_case_metadata": globals().get("target_case_metadata"),
    "prior_history": globals().get("prior_history_sampled", globals().get("prior_history_df", globals().get("prior_review_history_df"))),
    "training_prior_history": globals().get("training_prior_history_sampled"),
}

common_column_rows = []
for frame_name, frame in common_frame_candidates.items():
    common_column_rows.extend(summarize_common_frame(frame_name, frame))

common_columns_df = pd.DataFrame(common_column_rows)
if not common_columns_df.empty:
    (PROJECT_ROOT / "outputs" / STAGE).mkdir(parents=True, exist_ok=True)
    common_columns_df.to_csv(PROJECT_ROOT / "outputs" / STAGE / f"{CATEGORY_ID}_common_user_sampling_columns.csv", index=False, encoding="utf-8-sig")

common_contract = dict(COMMON_USER_SAMPLING_OUTPUT_CONTRACT)
common_contract["output_paths"] = {
    "common_user_sampling_contract": str(PROJECT_ROOT / "outputs" / STAGE / f"{CATEGORY_ID}_common_user_sampling_contract.json"),
    "common_user_sampling_columns": str(PROJECT_ROOT / "outputs" / STAGE / f"{CATEGORY_ID}_common_user_sampling_columns.csv"),
}
common_contract["frame_row_counts"] = {
    name: int(len(frame))
    for name, frame in common_frame_candidates.items()
    if frame is not None
}
common_contract["regime_history_scope"] = "strict_pre_target_excluding_target_item"
common_contract["stage1_profile_history_scope"] = (
    "strict_pre_target_and_before_training_cutoff_excluding_target_item"
)
common_contract["stage2_profile_history_scope"] = "strict_pre_target_excluding_target_item"
common_contract["common_critical_sampling_contract"] = COMMON_CRITICAL_SAMPLING_CONTRACT
common_contract["common_critical_sampling_contract_sha256"] = COMMON_CRITICAL_SAMPLING_CONTRACT_SHA256
common_contract["strict_prior_history_case_scope"] = "full_eligible_reserve_before_query_audit_and_balance"
common_contract["training_prior_history_case_scope"] = "full_eligible_reserve_before_query_audit_and_balance"
common_contract["notebook_03_output_role"] = "full_eligible_reserve_before_query_audit_and_balance"
common_contract["eligible_pool_regime_counts_before_query_balance"] = EXPECTED_ELIGIBLE_POOL_REGIME_COUNTS
common_contract["eligible_pool_total_n_before_query_balance"] = EXPECTED_ELIGIBLE_POOL_TOTAL_N
common_contract["query_balance_deferred_to_query_audit"] = True
common_contract["downstream_query_balance_n_per_regime"] = None
common_contract["non_cold_stage1_profile_required"] = True

(PROJECT_ROOT / "outputs" / STAGE).mkdir(parents=True, exist_ok=True)
with open(PROJECT_ROOT / "outputs" / STAGE / f"{CATEGORY_ID}_common_user_sampling_contract.json", "w", encoding="utf-8") as f:
    json.dump(common_contract, f, ensure_ascii=False, indent=2)

print("Saved common user sampling contract:", PROJECT_ROOT / "outputs" / STAGE / f"{CATEGORY_ID}_common_user_sampling_contract.json")
if not common_columns_df.empty:
    print("Saved common user sampling columns:", PROJECT_ROOT / "outputs" / STAGE / f"{CATEGORY_ID}_common_user_sampling_columns.csv")
    display(common_columns_df.head(30))

# ==== Validate Training-Safe Histories Against the Exported Cases ====
sampled_cases = pd.read_parquet(
    PROJECT_ROOT / "data/interim/user_regime_sampling/herbal_sampled_query_cases.parquet"
)
train_prior = pd.read_parquet(
    PROJECT_ROOT / "data/processed/user_sampling/herbal_user_prior_review_history_training.parquet"
)

print(sampled_cases["regime"].value_counts().reindex(["cold", "weak", "strong"], fill_value=0))

prior_counts = train_prior.groupby("case_id")["prior_item_id"].nunique()
check = sampled_cases[["case_id", "regime"]].copy()
check["train_prior_unique_items"] = check["case_id"].map(prior_counts).fillna(0).astype(int)

bad_non_cold = check[
    check["regime"].ne("cold")
    & check["train_prior_unique_items"].eq(0)
]

print("Notebook 03 non-cold zero-training-prior cases:", len(bad_non_cold))
if len(bad_non_cold):
    display(bad_non_cold.sort_values(["regime", "case_id"]))



Saved common user sampling contract: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/stage0_user_regime_sampling/herbal_common_user_sampling_contract.json
Saved common user sampling columns: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/stage0_user_regime_sampling/herbal_common_user_sampling_columns.csv


,frame_name,common_column,source_column,present,non_null_rows,unique_values
0,sampled_query_cases,case_id,case_id,True,28963,28963
1,sampled_query_cases,user_id,user_id,True,28963,28963
2,sampled_query_cases,target_item_id,target_parent_asin,True,28963,5828
3,sampled_query_cases,target_timestamp_ms,target_timestamp_ms,True,28963,28961
4,sampled_query_cases,target_review_datetime,target_review_datetime,True,28963,28961
5,sampled_query_cases,regime,regime,True,28963,3
6,sampled_query_cases,prior_count,prior_history_n,True,28963,82
7,sampled_query_cases,target_rank,target_rank_desc,True,28963,5
8,sampled_query_cases,target_selection_mode,target_selection_mode,True,28963,1
9,sampled_query_cases,n_pre_target_interactions,n_pre_target_interactions,True,28963,82


regime
cold      24590
weak       3703
strong      670
Name: count, dtype: int64
Notebook 03 non-cold zero-training-prior cases: 0


In [21]:
# ==== Audit Candidate Supply and Eligibility ====
diagnostic_cols = [
    "query_convertibility_failure_reason",
    "query_safe_token_count",
    "query_safe_signal_family_count",
    "query_safe_signal_total_count",
    "c_lite_query_safe_signal_family_count",
    "c_lite_query_safe_signal_total_count",
    "c_lite_token_fallback_pass",
    "c_lite_signal_pass",
    "target_rank_desc",
    "target_item_repeat_prior_flag",
]

available_diagnostic_cols = [col for col in diagnostic_cols if col in target_candidates.columns]

print("Target candidate diagnostics by convertibility:")
display(
    target_candidates
    .groupby("query_convertible_flag")[available_diagnostic_cols]
    .describe()
)

print("Failure reasons:")
display(
    target_candidates
    .loc[target_candidates["query_convertible_flag"].eq(0), "query_convertibility_failure_reason"]
    .value_counts()
    .head(20)
)

print("Convertible candidates by target rank:")
display(
    target_candidates
    .groupby(["target_rank_desc", "query_convertible_flag"])
    .size()
    .unstack(fill_value=0)
)

print("C-lite signal family counts:")
display(
    target_candidates[[
        "query_safe_signal_family_count",
        "query_safe_signal_total_count",
        "c_lite_query_safe_signal_family_count",
        "c_lite_query_safe_signal_total_count",
    ]]
    .describe()
)


Target candidate diagnostics by convertibility:


query_safe_token_count                                                      query_safe_signal_family_count                                               \
                                        count       mean        std  min   25%   50%   75%     max                          count      mean       std  min  25%  50%  75%  max   
query_convertible_flag                                                                                                                                                           
0                                     55242.0  26.678849  28.574272  0.0   9.0  18.0  34.0   620.0                        55242.0  0.011567  0.124016  0.0  0.0  0.0  0.0  4.0   
1                                     35246.0  61.241049  62.458116  3.0  23.0  43.0  77.0  1487.0                        35246.0  1.357799  0.634891  1.0  1.0  1.0  2.0  5.0   

                       query_safe_signal_total_count                                               c_lite_query_safe_signal_family_count                                               \
                                               count      mean       std  min  25%  50%  75%   max                                 count      mean       std  min  25%  50%  75%  max   
query_convertible_flag                                                                                                                                                                  
0                                            55242.0  0.013341  0.153828  0.0  0.0  0.0  0.0   7.0                               55242.0  0.008544  0.098317  0.0  0.0  0.0  0.0  3.0   
1                                            35246.0  1.610027  1.020166  1.0  1.0  1.0  2.0  12.0                               35246.0  1.018754  0.552958  0.0  1.0  1.0  1.0  3.0   

                       c_lite_query_safe_signal_total_count                                               c_lite_token_fallback_pass                                              c_lite_signal_pass  \
                                                      count      mean       std  min  25%  50%  75%   max                      count      mean       std  min  25%  50%  75%  max              count   
query_convertible_flag                                                                                                                                                                                 
0                                                   55242.0  0.010137  0.126095  0.0  0.0  0.0  0.0   5.0                    55242.0  0.976829  0.150447  0.0  1.0  1.0  1.0  1.0            55242.0   
1                                                   35246.0  1.222153  0.862953  0.0  1.0  1.0  2.0  10.0                    35246.0  1.000000  0.000000  1.0  1.0  1.0  1.0  1.0            35246.0   

                                                                    target_rank_desc                                              target_item_repeat_prior_flag                                     \
                            mean       std  min  25%  50%  75%  max            count      mean       std  min  25%  50%  75%  max                         count      mean       std  min  25%  50%   
query_convertible_flag                                                                                                                                                                               
0                       0.977083  0.149641  0.0  1.0  1.0  1.0  1.0          55242.0  1.167264  0.534748  1.0  1.0  1.0  1.0  5.0                       55242.0  0.001575  0.039654  0.0  0.0  0.0   
1                       1.000000  0.000000  1.0  1.0  1.0  1.0  1.0          35246.0  1.311525  0.783636  1.0  1.0  1.0  1.0  5.0                       35246.0  0.002951  0.054241  0.0  0.0  0.0   

                                  
                        75%  max  
query_convertible_flag            
0                       0.0  1.0  
1                       0.0  1.0

Failure reasons:


,count
query_convertibility_failure_reason,
low_signal_family_count,49697
short_review,4875
discontinued_item,670


Convertible candidates by target rank:


query_convertible_flag,0,1
target_rank_desc,,
1,48780,28835
2,4705,3774
3,988,1246
4,517,850
5,252,541


C-lite signal family counts:


,query_safe_signal_family_count,query_safe_signal_total_count,c_lite_query_safe_signal_family_count,c_lite_query_safe_signal_total_count
count,90488.000000,90488.000000,90488.000000,90488.000000
mean,0.535938,0.635267,0.402031,0.482230
std,0.772889,1.012942,0.606359,0.805656
min,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000
75%,1.000000,1.000000,1.000000,1.000000
max,5.000000,12.000000,3.000000,10.000000
